# Empirical analysis

This notebook contains only the code paths used by the current empirical tables and figures. Run all cells in order from a clean kernel.

## Environment and reproducibility seed


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_recall_fscore_support,
    precision_score, recall_score, roc_auc_score,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
import itertools
import math
import random
from pathlib import Path
from importlib import reload
from IPython.display import display
import sys

# Locate the repository root whether Jupyter starts from the root or code/.
_repository_candidates = [Path.cwd(), *Path.cwd().parents]
REPOSITORY_DIR = next(
    (
        path.resolve()
        for path in _repository_candidates
        if (path / 'data' / 'price.csv').exists()
    ),
    None,
)
if REPOSITORY_DIR is None:
    raise FileNotFoundError('Could not locate data/price.csv.')

CODE_DIR = REPOSITORY_DIR / 'code'
DATA_DIR = REPOSITORY_DIR / 'data'
for module_dir in [CODE_DIR, CODE_DIR / 'application']:
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from models.benchmark_classifiers import (
    BlockSparseMLPBinaryClassifier, FTTransformerBinaryClassifier,
    LSTMBinaryClassifier, MLPBinaryClassifier,
    ResNetTypeCNNBinaryClassifier,
)
from models.hierarchical_classifiers import build_hierarchical_cnn

In [ ]:

GLOBAL_SEED = 40

def set_seed(seed=GLOBAL_SEED, deterministic=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            torch.use_deterministic_algorithms(True)
    return seed

set_seed(GLOBAL_SEED)

## Data, event labels and rolling-window functions


In [ ]:
# =========================
# Data configuration
# =========================
PAPER_EVENT_FILE = DATA_DIR / "events.xlsx"
PAPER_CUTOFF_DATE = pd.Timestamp("2026-02-20")

# These are the three event families used by the hierarchical classifier.
PAPER_EVENT_FAMILIES = [
    "weather_natural_hazard",
    "geopolitical_security",
    "supply_policy_macro_financial",
]
# Stage 1 treats every listed family as the positive event class.
STAGE1_EVENT_CATEGORY_MAP = {family: 1 for family in PAPER_EVENT_FAMILIES}

price_df = pd.read_csv(DATA_DIR / "price.csv")
events_df = pd.read_excel(PAPER_EVENT_FILE)
observed_event_families = set(events_df['category'].dropna().unique())

price_df["observation_date"] = pd.to_datetime(price_df["observation_date"])
price_df = price_df.set_index("observation_date").sort_index()

events_df["start_date"] = pd.to_datetime(events_df["start_date"])
events_df["end_date"] = pd.to_datetime(events_df["end_date"])
events_df["event_date"] = pd.to_datetime(events_df["event_date"])

# Reserve observations on and after 2026-02-20 for the independent case study.
price_df = price_df.loc[price_df.index < PAPER_CUTOFF_DATE]
events_df = events_df.loc[events_df["start_date"] < PAPER_CUTOFF_DATE]


selected_commodities = list(price_df.columns)

# =========================
# Optional event impact filter
# =========================
# Remove low-impact event labels at runtime without modifying the Excel event table.
# Impact score = (max event-period price - min event-period price) / abs(mean event-period price).
ENABLE_EVENT_IMPACT_FILTER = True
EVENT_IMPACT_FILTER_MIN_RATIO = 0.20


def compute_event_price_range_ratio(row, price_df):
    commodity = row['commodity']
    if commodity not in price_df.columns:
        return np.nan, 0

    event_prices = price_df.loc[row['start_date']:row['end_date'], commodity].dropna()
    if event_prices.empty:
        return np.nan, 0

    price_mean = event_prices.mean()
    denominator = max(abs(price_mean), 1e-8)
    range_ratio = (event_prices.max() - event_prices.min()) / denominator
    return float(range_ratio), int(len(event_prices))


events_df = events_df.copy()
event_impact_values = events_df.apply(
    lambda row: compute_event_price_range_ratio(row, price_df),
    axis=1,
)
events_df['event_price_range_ratio'] = [item[0] for item in event_impact_values]
events_df['event_price_obs_count'] = [item[1] for item in event_impact_values]

if ENABLE_EVENT_IMPACT_FILTER:
    before_filter_n = len(events_df)
    low_impact_mask = (
        events_df['event_price_range_ratio'].notna()
        & (events_df['event_price_range_ratio'] < EVENT_IMPACT_FILTER_MIN_RATIO)
    )
    event_impact_filter_report = events_df.loc[
        low_impact_mask,
        [
            'event_id', 'commodity', 'category', 'subcategory',
            'start_date', 'end_date', 'event_price_range_ratio', 'event_price_obs_count'
        ]
    ].sort_values(['commodity', 'start_date']).reset_index(drop=True)

    events_df = events_df.loc[~low_impact_mask].copy()
    print(
        f'Event impact filter: removed {before_filter_n - len(events_df)} / {before_filter_n} events '
        f'with range_ratio < {EVENT_IMPACT_FILTER_MIN_RATIO:.2f}'
    )
else:
    event_impact_filter_report = pd.DataFrame()
    print('Event impact filter disabled.')

# =========================
# 2. Stage 1 label mapping
# =========================
category_map = STAGE1_EVENT_CATEGORY_MAP

# When several events match one window, select the primary event by curated
# intensity first, followed by the overlap ratio and event start date.
intensity_rank = {
    "low": 1,
    "medium": 2,
    "high": 3
}

# =========================
# 3. Group events by commodity
# =========================
events_by_commodity = {
    commodity: sub_df.to_dict("records")
    for commodity, sub_df in events_df.groupby("commodity")
}


In [ ]:
# Data utilities and rolling-window construction
def _event_window_overlap_ratio(window_start, window_end, event_start, event_end):
    overlap_start = max(window_start, event_start)
    overlap_end = min(window_end, event_end)
    if overlap_start > overlap_end:
        return 0.0, 0

    overlap_days = (overlap_end - overlap_start).days + 1
    window_days = (window_end - window_start).days + 1
    event_days = (event_end - event_start).days + 1
    ratio = overlap_days / max(1, min(window_days, event_days))
    return ratio, overlap_days


def build_window_dataset(
    window_len,
    price_df,
    events_by_commodity,
    category_map,
    intensity_rank,
    min_overlap_ratio=0.3,
    use_event_date_anchor=True,
):
    X_list, y_list, meta_list = [], [], []

    for col in price_df.columns:
        series = price_df[col].dropna()
        dates = series.index
        commodity_events = events_by_commodity.get(col, [])

        for i in range(window_len - 1, len(series)):
            window_start = dates[i - window_len + 1]
            window_end = dates[i]

            price_window = series.iloc[i - window_len + 1:i + 1].values.astype(float)

            relevant_events = []
            for ev in commodity_events:
                overlap_ratio, overlap_days = _event_window_overlap_ratio(
                    window_start,
                    window_end,
                    ev["start_date"],
                    ev["end_date"],
                )
                event_date_in_window = bool(
                    use_event_date_anchor
                    and pd.notna(ev.get("event_date"))
                    and (window_start <= ev["event_date"] <= window_end)
                )

                if (overlap_ratio >= min_overlap_ratio) or event_date_in_window:
                    ev_with_overlap = ev.copy()
                    ev_with_overlap["window_event_overlap_ratio"] = overlap_ratio
                    ev_with_overlap["window_event_overlap_days"] = overlap_days
                    ev_with_overlap["event_date_in_window"] = event_date_in_window
                    relevant_events.append(ev_with_overlap)

            if not relevant_events:
                label = 0
                chosen_event = None
            else:
                chosen_event = max(
                    relevant_events,
                    key=lambda x: (
                        intensity_rank.get(str(x["intensity"]).lower(), 0),
                        x["window_event_overlap_ratio"],
                        x["start_date"]
                    )
                )
                label = category_map.get(chosen_event["category"], 0)

            X_list.append(price_window)
            y_list.append(label)
            meta_list.append({
                "commodity": col,
                "window_start": window_start,
                "window_end": window_end,
                "series_start_position": int(i - window_len + 1),
                "series_end_position": int(i),
                "label": label,
                "event_id": chosen_event["event_id"] if chosen_event else None,
                "category": chosen_event["category"] if chosen_event else None,
                "n_events_in_window": len(relevant_events),
                "event_overlap_ratio": chosen_event["window_event_overlap_ratio"] if chosen_event else 0.0,
                "event_overlap_days": chosen_event["window_event_overlap_days"] if chosen_event else 0,
                "event_date_in_window": chosen_event["event_date_in_window"] if chosen_event else False,
            })

    return np.array(X_list), np.array(y_list), pd.DataFrame(meta_list)


def split_binary_dataset_by_commodity(
    X,
    y,
    meta_df,
    val_ratio=0.1,
    test_ratio=0.2,
    event_split_overrides=None,
    purge_shared_observations=False,
):
    y_binary = (y != 0).astype(int)
    required_position_columns = {'series_start_position', 'series_end_position'}
    if purge_shared_observations and not required_position_columns.issubset(meta_df.columns):
        raise ValueError('Re-run the window-construction cell before applying the overlap purge.')

    split_assignment = pd.Series(index=meta_df.index, dtype=object)
    for _, group in meta_df.groupby('commodity', sort=False):
        idx = group.index.to_numpy()
        split_idx = int(len(idx) * (1 - test_ratio))
        train_val_idx = idx[:split_idx]
        test_group_idx = idx[split_idx:]
        val_size = int(len(train_val_idx) * val_ratio)
        train_group_idx = train_val_idx[:-val_size]
        val_group_idx = train_val_idx[-val_size:]
        split_assignment.loc[train_group_idx] = 'train'
        split_assignment.loc[val_group_idx] = 'validation'
        split_assignment.loc[test_group_idx] = 'test'

    override_mask = pd.Series(False, index=meta_df.index)
    override_rows = []
    event_split_overrides = event_split_overrides or {}
    valid_split_names = {'train', 'validation', 'test'}
    for (commodity, event_id), target_split in event_split_overrides.items():
        if target_split not in valid_split_names:
            raise ValueError(f'Invalid target split {target_split!r} for {(commodity, event_id)}')
        event_mask = (
            meta_df['commodity'].astype(str).eq(str(commodity))
            & meta_df['event_id'].astype(str).eq(str(event_id))
        )
        if not event_mask.any():
            raise ValueError(f'Event override did not match any window for {(commodity, event_id)}')
        original_splits = sorted(split_assignment.loc[event_mask].unique())
        split_assignment.loc[event_mask] = target_split
        override_mask.loc[event_mask] = True
        override_rows.append({
            'commodity': commodity,
            'event_id': event_id,
            'original_splits': ', '.join(original_splits),
            'target_split': target_split,
            'event_windows': int(event_mask.sum()),
        })

    keep_mask = pd.Series(True, index=meta_df.index)
    if purge_shared_observations:
        normal_split_priority = {'train': 3, 'validation': 2, 'test': 1}
        for commodity, group in meta_df.groupby('commodity', sort=False):
            maximum_position = int(group['series_end_position'].max())
            covering_by_position = [[] for _ in range(maximum_position + 1)]
            for row_index, row in group[[
                'series_start_position', 'series_end_position'
            ]].iterrows():
                for position in range(int(row['series_start_position']), int(row['series_end_position']) + 1):
                    covering_by_position[position].append(row_index)

            for covering_indices in covering_by_position:
                if not covering_indices:
                    continue
                covering_splits = split_assignment.loc[covering_indices]
                if covering_splits.nunique() <= 1:
                    continue
                protected_splits = set(
                    split_assignment.loc[
                        [index for index in covering_indices if override_mask.loc[index]]
                    ]
                )
                if len(protected_splits) > 1:
                    raise ValueError(
                        f'Protected event windows conflict across splits for {commodity}'
                    )
                if protected_splits:
                    winning_split = next(iter(protected_splits))
                else:
                    winning_split = max(
                        set(covering_splits),
                        key=lambda split_name: normal_split_priority[split_name],
                    )
                losing_indices = [
                    index for index in covering_indices
                    if split_assignment.loc[index] != winning_split
                ]
                keep_mask.loc[losing_indices] = False

    kept_meta = meta_df.loc[keep_mask].copy()
    kept_splits = split_assignment.loc[keep_mask]
    train_idx = kept_meta.index[kept_splits.eq('train')].to_numpy(dtype=int)
    val_idx = kept_meta.index[kept_splits.eq('validation')].to_numpy(dtype=int)
    test_idx = kept_meta.index[kept_splits.eq('test')].to_numpy(dtype=int)

    if purge_shared_observations:
        for commodity, group in kept_meta.groupby('commodity', sort=False):
            split_position_sets = {}
            for split_name in ['train', 'validation', 'test']:
                split_rows = group.loc[kept_splits.loc[group.index].eq(split_name)]
                used_positions = set()
                for row in split_rows[[
                    'series_start_position', 'series_end_position'
                ]].itertuples(index=False):
                    used_positions.update(range(row.series_start_position, row.series_end_position + 1))
                split_position_sets[split_name] = used_positions
            assert split_position_sets['train'].isdisjoint(split_position_sets['validation'])
            assert split_position_sets['train'].isdisjoint(split_position_sets['test'])
            assert split_position_sets['validation'].isdisjoint(split_position_sets['test'])

    if override_rows:
        override_report_df = pd.DataFrame(override_rows)
        if len(override_report_df) <= 10:
            print('Manual binary event split overrides')
            display(override_report_df)
        else:
            print('Grouped event split assignments')
            display(override_report_df.groupby('target_split').agg(
                commodity_event_groups=('event_id', 'size'),
                event_windows=('event_windows', 'sum'),
            ).reindex(['train', 'validation', 'test']))
    if purge_shared_observations:
        print('Windows removed by the cross-split raw-observation purge:', int((~keep_mask).sum()))
        print('Raw-observation overlap audit passed.')

    return (
        X[train_idx], y_binary[train_idx],
        X[val_idx], y_binary[val_idx],
        X[test_idx], y_binary[test_idx],
        train_idx, val_idx, test_idx
    )


def downsample_binary_train_majority(X_train, y_train, target_neg_pos_ratio=1.5, random_seed=42):
    idx_neg = np.where(y_train == 0)[0]
    idx_pos = np.where(y_train == 1)[0]

    if len(idx_neg) == 0 or len(idx_pos) == 0:
        return X_train, y_train

    max_neg_keep = int(target_neg_pos_ratio * len(idx_pos))
    if len(idx_neg) <= max_neg_keep:
        return X_train, y_train

    rng = np.random.default_rng(random_seed)
    idx_neg_keep = rng.choice(idx_neg, size=max_neg_keep, replace=False)
    keep_idx = np.concatenate([idx_neg_keep, idx_pos])
    keep_idx.sort()

    return X_train[keep_idx], y_train[keep_idx]


def print_binary_split_stats(y_train, y_val, y_test):
    print("unique y_train:", np.unique(y_train))
    print("unique y_val:", np.unique(y_val))
    print("unique y_test:", np.unique(y_test))
    print("train 1 ratio:", (y_train == 1).mean())
    print("val   1 ratio:", (y_val == 1).mean())
    print("test  1 ratio:", (y_test == 1).mean())


def get_commodity_binary_splits(commodity):
    """Return one commodity's downsampled training and unchanged evaluation splits."""
    binary_labels = (np.asarray(y) != 0).astype(int)
    split_indices = {
        'train': np.asarray(train_idx_bin, dtype=int),
        'validation': np.asarray(val_idx_bin, dtype=int),
        'test': np.asarray(test_idx_bin, dtype=int),
    }
    split_data = {}
    for split_name, candidate_indices in split_indices.items():
        commodity_mask = (
            meta_df.loc[candidate_indices, 'commodity'].astype(str).to_numpy()
            == str(commodity)
        )
        indices = candidate_indices[commodity_mask]
        labels = binary_labels[indices]
        class_counts = np.bincount(labels, minlength=2)
        if np.any(class_counts == 0):
            raise ValueError(
                f'{commodity} {split_name} lacks one binary class: {class_counts.tolist()}'
            )
        split_data[split_name] = {
            'indices': indices,
            'X': np.asarray(X[indices]).copy(),
            'y': labels.copy(),
        }

    n_train_before_downsampling = len(split_data['train']['y'])
    if ENABLE_TRAIN_DOWNSAMPLE:
        train_X, train_y = downsample_binary_train_majority(
            split_data['train']['X'],
            split_data['train']['y'],
            target_neg_pos_ratio=TARGET_NEG_POS_RATIO,
            random_seed=DOWNSAMPLE_RANDOM_SEED,
        )
        split_data['train'] = {'X': train_X, 'y': train_y}

    split_data['n_train_before_downsampling'] = n_train_before_downsampling
    return split_data


## Shared CNN construction and training utilities


In [ ]:
# CNN
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # X: (N, L) -> (N, 1, L)
        self.X = torch.tensor(X[:, None, :], dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]    

def make_binary_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size_train=128, batch_size_eval=256):
    train_loader = DataLoader(
        TimeSeriesDataset(X_train, y_train),
        batch_size=batch_size_train, shuffle=True, num_workers=0,
    )
    val_loader = DataLoader(
        TimeSeriesDataset(X_val, y_val),
        batch_size=batch_size_eval, shuffle=False, num_workers=0,
    )
    test_loader = DataLoader(
        TimeSeriesDataset(X_test, y_test),
        batch_size=batch_size_eval, shuffle=False, num_workers=0,
    )
    return train_loader, val_loader, test_loader

# The paper CNN is constructed from the shared parameterised framework in
# code/models. Loss functions and training procedures remain below.
def make_class_weights(y_train):
    class_counts = pd.Series(y_train).value_counts().sort_index()
    total = class_counts.sum()
    n_classes = len(class_counts)
    weights = total / (n_classes * class_counts)
    return torch.tensor(weights.values, dtype=torch.float32)


def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=30):
    model.to(device)
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        print(
            f"Epoch {epoch+1:03d}/{epochs} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_loss:.4f}"
        )

def predict_model(model, data_loader, device):
    model.eval()
    model.to(device)

    all_logits = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1)

            all_logits.append(logits.cpu())
            all_preds.append(preds.cpu())
            all_targets.append(y_batch)

    all_logits = torch.cat(all_logits).numpy()
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    return all_logits, all_preds, all_targets


@torch.no_grad()
def predict_windows(model, X_values, device, batch_size=256):
    """Return binary predictions and event probabilities for window arrays."""
    X_values = np.asarray(X_values, dtype=np.float32)
    if len(X_values) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)
    loader = DataLoader(
        TimeSeriesDataset(X_values, np.zeros(len(X_values), dtype=int)),
        batch_size=batch_size, shuffle=False, num_workers=0,
    )
    model.eval()
    logits = torch.cat([
        model(X_batch.to(device)).cpu() for X_batch, _ in loader
    ])
    probabilities = torch.softmax(logits, dim=1)[:, 1].numpy()
    predictions = torch.argmax(logits, dim=1).numpy().astype(int)
    return predictions, probabilities

#imbalance handling
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B, C)
        # targets: (B,)

        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        probs = torch.softmax(logits, dim=1)
        pt = probs[torch.arange(len(targets), device=logits.device), targets]

        focal_weight = (1 - pt) ** self.gamma

        if self.alpha is None:
            alpha_t = 1.0
        else:
            alpha = self.alpha.to(logits.device)
            alpha_t = alpha[targets]   # Select the class weight for each observation.

        loss = alpha_t * focal_weight * ce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss
        


## Window length and Stage 1 dataset

The default path reproduces the recorded results with an 80-observation window. The optional audit reconstructs the intended validation-only search under the current leakage controls.


In [ ]:
# The reported paper tables use L = 80. The optional audit compares candidate
# lengths from 30 to 90 with the event-safe, per-commodity split and Stage-1
# training protocol.

PAPER_WINDOW_LENGTH = 80
WINDOW_LENGTH_CANDIDATES = [30, 40, 50, 60, 70, 80, 90]

# False reproduces the stored paper results with their recorded length of 80.
# Set this to True to rerun the scientifically valid validation-only audit first.
RUN_WINDOW_LENGTH_AUDIT = False
WINDOW_LENGTH_AUDIT_EPOCHS = 30

KNOWN_BINARY_EVENT_SPLIT_OVERRIDES = {
    ('DGASNYH', 'EVT24846'): 'train',
    ('DRGASLA', 'EVT_DRGASLA_2014_PRICE_CRASH'): 'validation',
    ('DCOILBRENTEU', 'EVT24886'): 'test',
    ('DDFUELUSGULF', 'EVT_DDFUELUSGULF_RUSSIA_UKRAINE_2022'): 'test',
}
BINARY_VAL_RATIO = 0.3
BINARY_TEST_RATIO = 0.2
ENABLE_TRAIN_DOWNSAMPLE = True
TARGET_NEG_POS_RATIO = 1.5
DOWNSAMPLE_RANDOM_SEED = 42


def build_binary_event_split_overrides(meta):
    preliminary_split = pd.Series(index=meta.index, dtype=object)
    for _, commodity_windows in meta.groupby('commodity', sort=False):
        commodity_idx = commodity_windows.index.to_numpy()
        test_start = int(len(commodity_idx) * (1 - BINARY_TEST_RATIO))
        train_validation_idx = commodity_idx[:test_start]
        test_idx = commodity_idx[test_start:]
        validation_size = int(len(train_validation_idx) * BINARY_VAL_RATIO)
        train_idx = train_validation_idx[:-validation_size]
        validation_idx = train_validation_idx[-validation_size:]
        preliminary_split.loc[train_idx] = 'train'
        preliminary_split.loc[validation_idx] = 'validation'
        preliminary_split.loc[test_idx] = 'test'

    event_mask = meta['event_id'].notna() & meta['label'].ne(0)
    event_frame = meta.loc[event_mask, ['commodity', 'event_id']].copy()
    event_frame['preliminary_split'] = preliminary_split.loc[event_frame.index].to_numpy()
    split_order = {'train': 0, 'validation': 1, 'test': 2}
    overrides = {}
    for (commodity, event_id), event_windows in event_frame.groupby(
        ['commodity', 'event_id'], sort=False
    ):
        observed = event_windows['preliminary_split'].dropna().unique().tolist()
        if len(observed) > 1:
            overrides[(str(commodity), str(event_id))] = max(
                observed, key=split_order.get
            )

    available_pairs = set(zip(
        event_frame['commodity'].astype(str),
        event_frame['event_id'].astype(str),
    ))
    overrides.update({
        (str(commodity), str(event_id)): target_split
        for (commodity, event_id), target_split in KNOWN_BINARY_EVENT_SPLIT_OVERRIDES.items()
        if (str(commodity), str(event_id)) in available_pairs
    })
    return overrides


@torch.no_grad()
def validation_metrics(model, loader, device):
    logits, predictions, truth = predict_model(model, loader, device)
    probability = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    return {
        'accuracy': accuracy_score(truth, predictions),
        'precision': precision_score(truth, predictions, zero_division=0),
        'recall': recall_score(truth, predictions, zero_division=0),
        'f1': f1_score(truth, predictions, zero_division=0),
        'auc': (
            roc_auc_score(truth, probability)
            if len(np.unique(truth)) == 2 else np.nan
        ),
    }


def run_event_safe_window_length_audit(candidate_lengths, epochs=30):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    rows = []
    for candidate_length in candidate_lengths:
        print(f'Window-length audit for T={candidate_length}')
        X_candidate, y_candidate, meta_candidate = build_window_dataset(
            candidate_length,
            price_df,
            events_by_commodity,
            category_map,
            intensity_rank,
        )
        candidate_overrides = build_binary_event_split_overrides(meta_candidate)
        (
            _, _, _, _, _, _,
            train_idx, validation_idx, _test_idx,
        ) = split_binary_dataset_by_commodity(
            X_candidate,
            y_candidate,
            meta_candidate,
            val_ratio=BINARY_VAL_RATIO,
            test_ratio=BINARY_TEST_RATIO,
            event_split_overrides=candidate_overrides,
            purge_shared_observations=True,
        )
        y_binary = (np.asarray(y_candidate) != 0).astype(int)

        commodity_rows = []
        for commodity in selected_commodities:
            train_commodity_idx = train_idx[
                meta_candidate.loc[train_idx, 'commodity'].eq(commodity).to_numpy()
            ]
            validation_commodity_idx = validation_idx[
                meta_candidate.loc[validation_idx, 'commodity'].eq(commodity).to_numpy()
            ]
            if len(train_commodity_idx) == 0 or len(validation_commodity_idx) == 0:
                raise RuntimeError(
                    f'Missing train or validation windows for {commodity} at T={candidate_length}.'
                )

            X_train = X_candidate[train_commodity_idx]
            y_train = y_binary[train_commodity_idx]
            X_validation = X_candidate[validation_commodity_idx]
            y_validation = y_binary[validation_commodity_idx]
            if ENABLE_TRAIN_DOWNSAMPLE:
                X_train, y_train = downsample_binary_train_majority(
                    X_train,
                    y_train,
                    target_neg_pos_ratio=TARGET_NEG_POS_RATIO,
                    random_seed=DOWNSAMPLE_RANDOM_SEED,
                )

            set_seed(GLOBAL_SEED)
            train_loader, validation_loader, _ = make_binary_loaders(
                X_train,
                y_train,
                X_validation,
                y_validation,
                X_validation,
                y_validation,
                batch_size_train=128,
                batch_size_eval=256,
            )
            class_weights = make_class_weights(y_train)
            model = build_hierarchical_cnn(
                input_length=candidate_length,
                num_classes=2,
                dropout=0.3,
            )
            criterion = FocalLoss(alpha=class_weights.to(device), gamma=2)
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
            train_model(
                model,
                train_loader,
                validation_loader,
                criterion,
                optimizer,
                device,
                epochs=epochs,
            )
            metrics = validation_metrics(model, validation_loader, device)
            commodity_rows.append({'commodity': commodity, **metrics})

        commodity_df = pd.DataFrame(commodity_rows)
        macro_metrics = commodity_df[
            ['accuracy', 'precision', 'recall', 'f1', 'auc']
        ].mean(numeric_only=True)
        rows.append({
            'window_length': candidate_length,
            'n_windows': len(X_candidate),
            **{f'macro_{name}': float(value) for name, value in macro_metrics.items()},
        })

    results = pd.DataFrame(rows).sort_values(
        ['macro_f1', 'macro_auc', 'window_length'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return results


if RUN_WINDOW_LENGTH_AUDIT:
    window_length_audit_results = run_event_safe_window_length_audit(
        WINDOW_LENGTH_CANDIDATES,
        epochs=WINDOW_LENGTH_AUDIT_EPOCHS,
    )
    display(window_length_audit_results)
    selected_window_length = int(window_length_audit_results.iloc[0]['window_length'])
    if selected_window_length != PAPER_WINDOW_LENGTH:
        raise RuntimeError(
            'The corrected validation audit did not select the recorded paper length of 80. '
            'Stop here and reconcile the manuscript and reported results before testing.'
        )
else:
    window_length_audit_results = None
    selected_window_length = PAPER_WINDOW_LENGTH
    print(
        'Window-length audit skipped. Reproducing the stored paper results with '
        f'T={selected_window_length}.'
    )

L = selected_window_length
X, y, meta_df = build_window_dataset(
    L,
    price_df,
    events_by_commodity,
    category_map,
    intensity_rank,
)
print('Selected window length', L)
print('X shape', X.shape)
print('y shape', y.shape)
display(meta_df.head())
display(meta_df.groupby('commodity')['label'].value_counts().sort_index())


## Examples for the event-labelling figure

This optional cell recreates the event-period figures used as labelling examples. Change `EVENT_EXAMPLE_COMMODITY` and its display settings to preview the geopolitical and supply-financial periods for another commodity. An individual example may override the default commodity, as the weather panel does for Henry Hub natural gas. The cell reloads the complete price file and does not change the six-commodity estimation sample or retrain any model. Figure saving is disabled by default.


In [ ]:
# Selectable version of the original WTI event-example plotting code.
# This cell is independent of model fitting and does not alter price_df or events_df.
import matplotlib.dates as mdates
import textwrap

EVENT_EXAMPLE_COMMODITY = 'DCOILBRENTEU'  # Replace with another price.csv column.
EVENT_EXAMPLE_Y_LABEL = 'Dollars per Barrel'
EVENT_EXAMPLE_WINDOW_LENGTH = int(globals().get('PAPER_WINDOW_LENGTH', 80))
EVENT_EXAMPLE_MIN_OVERLAP_RATIO = 0.30
EVENT_EXAMPLE_USE_EVENT_DATE_ANCHOR = True
EVENT_EXAMPLE_MONTHS_BEFORE = 12
EVENT_EXAMPLE_MONTHS_AFTER = 12

EVENT_EXAMPLE_STYLES = {
    'geopolitical_security': {
        'pale': '#f3a1a1', 'strong': '#dc2626', 'shade': '#ef4444',
    },
    'supply_policy_macro_financial': {
        'pale': '#9ed9a5', 'strong': '#16a34a', 'shade': '#22c55e',
    },
    'weather_natural_hazard': {
        'pale': '#d8b4fe', 'strong': '#7c3aed', 'shade': '#a855f7',
    },
}

EVENT_EXAMPLES = [
    {
        'key': 'war',
        'event_group_name': 'Gulf War / Iraq invades Kuwait',
        'start': '1990-08-02', 'end': '1991-02-28',
        'event_date': '1991-01-16',
        'category': 'geopolitical_security',
        'annotation_x_offset_days': 20, 'annotation_y_offset': -10.0,
    },
    {
        'key': 'supply',
        'event_group_name': 'COVID-19 demand collapse',
        'start': '2020-01-06', 'end': '2020-04-21',
        'event_date': '2020-03-11',
        'category': 'supply_policy_macro_financial',
        'annotation_x_offset_days': 25, 'annotation_y_offset': -50.0,
    },
    {
        'key': 'weather',
        'commodity': 'DHHNGSP',
        'y_axis_label': 'Dollars per Million BTU',
        'event_group_name': 'Winter Storm Uri / Texas freeze',
        'start': '2021-02-13', 'end': '2021-02-19',
        'event_date': '2021-02-17',
        'category': 'weather_natural_hazard',
        'annotation_x_offset_days': 35, 'annotation_y_offset': -10.0,
    },
]


def load_event_example_price_series(price_file, commodity):
    full_price_df = pd.read_csv(price_file)
    full_price_df['observation_date'] = pd.to_datetime(full_price_df['observation_date'])
    if commodity not in full_price_df.columns:
        raise ValueError(f'{commodity!r} is not a column in {price_file}.')
    return (
        full_price_df[['observation_date', commodity]]
        .dropna()
        .rename(columns={'observation_date': 'date', commodity: 'price'})
        .sort_values('date')
        .reset_index(drop=True)
    )


def event_example_positive_window_coverage(
    price, event_start, event_date, event_end, window_length,
):
    positive_windows = []
    for row_index in range(len(price) - window_length + 1):
        window_start = price.loc[row_index, 'date']
        window_end = price.loc[row_index + window_length - 1, 'date']
        overlap_start = max(window_start, event_start)
        overlap_end = min(window_end, event_end)
        overlap_days = max(0, (overlap_end - overlap_start).days + 1)
        window_days = (window_end - window_start).days + 1
        event_days = (event_end - event_start).days + 1
        overlap_ratio = overlap_days / max(1, min(window_days, event_days))
        contains_event_date = bool(
            EVENT_EXAMPLE_USE_EVENT_DATE_ANCHOR
            and window_start <= event_date <= window_end
        )
        if overlap_ratio >= EVENT_EXAMPLE_MIN_OVERLAP_RATIO or contains_event_date:
            positive_windows.append((window_start, window_end))
    if not positive_windows:
        return None, None
    return (
        min(start for start, _ in positive_windows),
        max(end for _, end in positive_windows),
    )


def plot_selectable_event_example(example):
    commodity = example.get('commodity', EVENT_EXAMPLE_COMMODITY)
    y_axis_label = example.get('y_axis_label', EVENT_EXAMPLE_Y_LABEL)
    price = load_event_example_price_series(DATA_DIR / 'price.csv', commodity)
    event_start = pd.Timestamp(example['start'])
    event_date = pd.Timestamp(example['event_date'])
    event_end = pd.Timestamp(example['end'])
    plot_start = event_start - pd.DateOffset(months=EVENT_EXAMPLE_MONTHS_BEFORE)
    plot_end = event_end + pd.DateOffset(months=EVENT_EXAMPLE_MONTHS_AFTER)
    plot_df = price.loc[price['date'].between(plot_start, plot_end)].copy()
    if plot_df.empty:
        raise ValueError('No price observations fall in the requested plot period.')

    coverage_start, coverage_end = event_example_positive_window_coverage(
        price, event_start, event_date, event_end, EVENT_EXAMPLE_WINDOW_LENGTH
    )
    coverage_df = plot_df.loc[
        plot_df['date'].between(max(plot_start, coverage_start), min(plot_end, coverage_end))
    ] if coverage_start is not None else plot_df.iloc[0:0]
    event_df = plot_df.loc[plot_df['date'].between(event_start, event_end)]
    style = EVENT_EXAMPLE_STYLES[example['category']]

    fig, ax = plt.subplots(figsize=(10.5, 5.6), dpi=220)
    ax.plot(plot_df['date'], plot_df['price'], color='#263746', linewidth=1.45, label='Price')
    ax.axvspan(event_start, event_end, color=style['shade'], alpha=0.055)
    ax.axvline(event_start, color=style['shade'], linestyle='--', linewidth=0.9, alpha=0.72)
    ax.axvline(event_end, color=style['shade'], linestyle='--', linewidth=0.9, alpha=0.72)
    if not coverage_df.empty:
        ax.plot(
            coverage_df['date'], coverage_df['price'], color=style['pale'],
            linewidth=4.4, alpha=0.74, solid_capstyle='round',
            label=f'Labelled-window coverage (L={EVENT_EXAMPLE_WINDOW_LENGTH})',
        )
    ax.plot(
        event_df['date'], event_df['price'], color=style['strong'],
        linewidth=2.0, alpha=0.92, solid_capstyle='round', label='Actual event span',
    )

    event_label = textwrap.fill(example['event_group_name'], width=30)
    event_label += f'\n{event_start:%Y-%m-%d} to {event_end:%Y-%m-%d}'
    ax.text(
        event_end + pd.DateOffset(days=example['annotation_x_offset_days']),
        float(plot_df['price'].max()) + example['annotation_y_offset'],
        event_label, fontsize=12, ha='left', va='top',
        bbox=dict(
            boxstyle='round,pad=0.25', facecolor='white',
            edgecolor=style['pale'], alpha=0.95,
        ),
    )
    ax.set_xlabel('Date', fontsize=14)
    ax.set_ylabel(y_axis_label, fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    ax.grid(True, color='#e5e7eb', linewidth=0.8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(
        loc='upper right', frameon=True, facecolor='white',
        edgecolor='#d1d5db', fontsize=12,
    )
    fig.tight_layout()

    plt.show()


for example in EVENT_EXAMPLES:
    plot_selectable_event_example(example)
print('Default commodity:', EVENT_EXAMPLE_COMMODITY)
print('Weather example commodity:', EVENT_EXAMPLES[-1].get('commodity'))


## Leakage-safe Stage 1 split


In [ ]:
# Leakage-safe binary split
# Per-commodity split with automatic protection for boundary-crossing events.
# Every commodity-event group stays in one subset. Known manual assignments take priority;
# all other boundary-crossing events are assigned wholly to the later subset.
# Windows sharing any raw price observation across splits are then purged.
BINARY_EVENT_SPLIT_OVERRIDES = build_binary_event_split_overrides(meta_df)

skipped_manual_overrides = sorted(
    set(KNOWN_BINARY_EVENT_SPLIT_OVERRIDES).difference(BINARY_EVENT_SPLIT_OVERRIDES)
)
if skipped_manual_overrides:
    print(f'Skipped manual event assignments outside the current sample {skipped_manual_overrides}')

X_train_bin2, y_train_bin2, X_val_bin, y_val_bin, X_test_scaled, y_test_bin, train_idx_bin, val_idx_bin, test_idx_bin = split_binary_dataset_by_commodity(
    X,
    y,
    meta_df,
    val_ratio=BINARY_VAL_RATIO,
    test_ratio=BINARY_TEST_RATIO,
    event_split_overrides=BINARY_EVENT_SPLIT_OVERRIDES,
    purge_shared_observations=True,
)

print(X_train_bin2.shape, y_train_bin2.shape)
# Optionally downsample the non-event class in the training set only.
if ENABLE_TRAIN_DOWNSAMPLE:
    print('Before downsampling train class counts:')
    print(pd.Series(y_train_bin2).value_counts().sort_index())
    X_train_bin2, y_train_bin2 = downsample_binary_train_majority(
        X_train_bin2,
        y_train_bin2,
        target_neg_pos_ratio=TARGET_NEG_POS_RATIO,
        random_seed=DOWNSAMPLE_RANDOM_SEED
    )
    print('After downsampling train class counts:')
    print(pd.Series(y_train_bin2).value_counts().sort_index())

print(X_train_bin2.shape, y_train_bin2.shape)
print(X_val_bin.shape, y_val_bin.shape)
print(X_test_scaled.shape, y_test_bin.shape)
print_binary_split_stats(y_train_bin2, y_val_bin, y_test_bin)



## Stage 1 binary CNN results


In [ ]:
# Train one independent binary CNN per commodity on the event-safe split above.
# The architecture, downsampling, focal loss, seed, epochs, and test threshold
# match the pooled binary experiment so that parameter sharing is the only change.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Independent-model device:', device)

def run_independent_binary_cnns(commodities=selected_commodities, epochs=30):
    result_rows = []
    prediction_parts = []

    for commodity_number, commodity in enumerate(commodities, start=1):
        print(f'\nIndependent binary CNN {commodity_number}/{len(commodities)}  {commodity}')
        data = get_commodity_binary_splits(commodity)

        set_seed(GLOBAL_SEED)
        train_loader, validation_loader, test_loader = make_binary_loaders(
            data['train']['X'],
            data['train']['y'],
            data['validation']['X'],
            data['validation']['y'],
            data['test']['X'],
            data['test']['y'],
            batch_size_train=128,
            batch_size_eval=256,
        )
        commodity_class_weights = make_class_weights(data['train']['y'])
        commodity_model = build_hierarchical_cnn(
            input_length=data['train']['X'].shape[1],
            num_classes=2,
            dropout=0.3,
        )
        commodity_criterion = FocalLoss(
            alpha=commodity_class_weights.to(device),
            gamma=2,
        )
        commodity_optimizer = torch.optim.Adam(commodity_model.parameters(), lr=1e-3)
        train_model(
            commodity_model,
            train_loader,
            validation_loader,
            commodity_criterion,
            commodity_optimizer,
            device,
            epochs=epochs,
        )

        commodity_logits, commodity_prediction, commodity_truth = predict_model(
            commodity_model,
            test_loader,
            device,
        )
        commodity_probability = torch.softmax(
            torch.tensor(commodity_logits), dim=1
        )[:, 1].numpy()
        commodity_auc = (
            roc_auc_score(commodity_truth, commodity_probability)
            if len(np.unique(commodity_truth)) == 2
            else np.nan
        )
        commodity_cm = confusion_matrix(
            commodity_truth, commodity_prediction, labels=[0, 1]
        )
        tn, fp, fn, tp = commodity_cm.ravel()
        metrics = {
            'commodity': commodity,
            'n_train_before_downsampling': data['n_train_before_downsampling'],
            'n_train_fit': len(data['train']['y']),
            'n_validation': len(data['validation']['y']),
            'n_test': len(commodity_truth),
            'positive_ratio': float(np.mean(commodity_truth)),
            'accuracy': accuracy_score(commodity_truth, commodity_prediction),
            'precision': precision_score(commodity_truth, commodity_prediction, zero_division=0),
            'recall': recall_score(commodity_truth, commodity_prediction, zero_division=0),
            'f1': f1_score(commodity_truth, commodity_prediction, zero_division=0),
            'auc': commodity_auc,
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn),
            'tp': int(tp),
        }
        result_rows.append(metrics)
        prediction_frame = meta_df.loc[data['test']['indices']].reset_index(drop=True).copy()
        prediction_frame['true_label'] = commodity_truth
        prediction_frame['pred_label'] = commodity_prediction
        prediction_frame['pred_prob_event'] = commodity_probability
        prediction_parts.append(prediction_frame)
        print(
            '  test F1', round(metrics['f1'], 4),
            'test AUC', round(metrics['auc'], 4),
        )

    metrics_df = pd.DataFrame(result_rows).sort_values('commodity').reset_index(drop=True)
    predictions_df = pd.concat(prediction_parts, ignore_index=True)
    macro_df = metrics_df[
        ['accuracy', 'precision', 'recall', 'f1', 'auc']
    ].mean(numeric_only=True).to_frame('macro_avg').T
    pooled_metrics = pd.DataFrame([{
        'accuracy': accuracy_score(predictions_df['true_label'], predictions_df['pred_label']),
        'precision': precision_score(
            predictions_df['true_label'], predictions_df['pred_label'], zero_division=0
        ),
        'recall': recall_score(
            predictions_df['true_label'], predictions_df['pred_label'], zero_division=0
        ),
        'f1': f1_score(
            predictions_df['true_label'], predictions_df['pred_label'], zero_division=0
        ),
        'auc': roc_auc_score(
            predictions_df['true_label'], predictions_df['pred_prob_event']
        ),
    }], index=['all_test_windows'])
    return metrics_df, macro_df, pooled_metrics, predictions_df


(
    independent_binary_metrics_df,
    independent_binary_macro_df,
    independent_binary_pooled_df,
    independent_binary_predictions_df,
) = run_independent_binary_cnns()

print('\nIndependent CNN results by commodity')
display(independent_binary_metrics_df)
print('\nEqual-weight macro average across commodities')
display(independent_binary_macro_df)
print('\nMetrics over all concatenated test windows')
display(independent_binary_pooled_df)


## Stage 1 representative TP, TN, FP and FN figures


In [ ]:
# Visualize one selectable confusion-matrix example: TP, TN, FP, or FN
CONFUSION_CASE_LABELS = {
    'TP': 'True Positive',
    'TN': 'True Negative',
    'FP': 'False Positive',
    'FN': 'False Negative',
}

# Keep all true-event intervals green, matching the figures used in the paper.
EVENT_HIGHLIGHT_COLOR = '#16a34a'

PRED_COLORS = {
    1: '#dc3545',
    0: '#6c757d',
}

PRICE_YLABEL_BY_COMMODITY = {
    'DHHNGSP': 'Price ($/MMBtu)',
}
DEFAULT_PRICE_YLABEL = 'Price'


def build_confusion_example_detail_df():
    required_names = ['independent_binary_predictions_df']
    missing_names = [name for name in required_names if name not in globals()]
    if missing_names:
        raise RuntimeError(
            'Run the independent per-commodity binary CNN cell first. '
            f'Missing: {missing_names}'
        )

    detail_df = independent_binary_predictions_df.copy().reset_index(drop=True)
    required_columns = {
        'commodity', 'window_start', 'window_end', 'true_label',
        'pred_label', 'pred_prob_event',
    }
    missing_columns = sorted(required_columns.difference(detail_df.columns))
    if missing_columns:
        raise RuntimeError(f'Independent prediction table lacks columns: {missing_columns}')

    detail_df['window_start'] = pd.to_datetime(detail_df['window_start'])
    detail_df['window_end'] = pd.to_datetime(detail_df['window_end'])
    detail_df['test_sample_index'] = np.arange(len(detail_df))
    detail_df['true_label'] = detail_df['true_label'].astype(int)
    detail_df['pred_label'] = detail_df['pred_label'].astype(int)
    detail_df['confusion_case'] = np.select(
        [
            (detail_df['true_label'] == 1) & (detail_df['pred_label'] == 1),
            (detail_df['true_label'] == 0) & (detail_df['pred_label'] == 0),
            (detail_df['true_label'] == 0) & (detail_df['pred_label'] == 1),
            (detail_df['true_label'] == 1) & (detail_df['pred_label'] == 0),
        ],
        ['TP', 'TN', 'FP', 'FN'],
        default='unknown'
    )
    return detail_df


def select_confusion_example(
    detail_df, case, strategy='representative', sample_n=None, window_key=None,
):
    if window_key is not None:
        commodity = str(window_key['commodity'])
        window_start = pd.Timestamp(window_key['window_start'])
        window_end = pd.Timestamp(window_key['window_end'])
        exact_match = detail_df.loc[
            detail_df['commodity'].astype(str).eq(commodity)
            & detail_df['window_start'].eq(window_start)
            & detail_df['window_end'].eq(window_end)
        ].copy()
        if exact_match.empty:
            raise KeyError(
                f'Selected {case} window is not in the independent-CNN test set: '
                f'{commodity}, {window_start:%Y-%m-%d} to {window_end:%Y-%m-%d}.'
            )
        row = exact_match.iloc[0]
        observed_case = str(row['confusion_case'])
        if observed_case != case:
            raise ValueError(
                f'Selected {case} window is classified as {observed_case} by the '
                f'independent CNN: {commodity}, {window_start:%Y-%m-%d} to {window_end:%Y-%m-%d}.'
            )
        return row

    subset = detail_df[detail_df['confusion_case'] == case].copy()
    if subset.empty:
        return None

    if sample_n is not None:
        subset = subset.sort_values('test_sample_index').reset_index(drop=True)
        if sample_n < 1 or sample_n > len(subset):
            raise IndexError(f'sample_n must be between 1 and {len(subset)} for case={case}.')
        return subset.iloc[sample_n - 1]

    if strategy == 'representative':
        if case in ['TP', 'FP']:
            return subset.sort_values('pred_prob_event', ascending=False).iloc[0]
        if case in ['TN', 'FN']:
            return subset.sort_values('pred_prob_event', ascending=True).iloc[0]
    elif strategy == 'first':
        return subset.sort_values('test_sample_index').iloc[0]
    else:
        raise ValueError("strategy must be 'representative' or 'first'.")


def _events_visible_in_range(commodity, plot_start, plot_end):
    events = []
    for ev in events_by_commodity.get(commodity, []):
        if (plot_start <= ev['end_date']) and (plot_end >= ev['start_date']):
            events.append(ev)
    return events


def _event_name(ev):
    return str(ev.get('subcategory', ev.get('category', 'event')))


def _wrap_text(text, width=54):
    words = str(text).split()
    lines = []
    current = []
    current_len = 0
    for word in words:
        extra = 1 if current else 0
        if current and current_len + extra + len(word) > width:
            lines.append(' '.join(current))
            current = [word]
            current_len = len(word)
        else:
            current.append(word)
            current_len += extra + len(word)
    if current:
        lines.append(' '.join(current))
    return '\n'.join(lines)


def _plot_single_confusion_example(
    ax,
    row,
    context_days=20,
    legend_fontsize=10,
    event_text_fontsize=9,
    axis_label_fontsize=11,
    tick_label_fontsize=10,
    title_fontsize=11,
    event_text_x_offset_days=0,
    event_text_y_base=0.08,
    event_text_y_step=0.10,
    event_text_wrap_width=34,
    y_limits=None,
    y_axis_label=None,
    legend_loc='best',
):
    if row is None:
        ax.axis('off')
        ax.text(0.5, 0.5, 'No sample found', ha='center', va='center', transform=ax.transAxes)
        return None

    commodity = row['commodity']
    case = row['confusion_case']

    price_series = price_df[commodity].dropna()
    plot_start = row['window_start'] - pd.Timedelta(days=context_days)
    plot_end = row['window_end'] + pd.Timedelta(days=context_days)
    visible_prices = price_series.loc[plot_start:plot_end]

    ax.plot(visible_prices.index, visible_prices.values, color='#263746', linewidth=1.4, label='Price')

    visible_events = _events_visible_in_range(commodity, plot_start, plot_end)
    event_summary = []
    used_event_labels = set()
    if y_limits is None:
        y_min, y_max = float(visible_prices.min()), float(visible_prices.max())
    else:
        y_min, y_max = y_limits
        ax.set_ylim(y_min, y_max)
    y_range = y_max - y_min if y_max > y_min else 1.0

    for event_idx, ev in enumerate(visible_events):
        color = EVENT_HIGHLIGHT_COLOR
        event_start = max(ev['start_date'], plot_start)
        event_end = min(ev['end_date'], plot_end)
        label = 'Event' if 'Event' not in used_event_labels else None
        used_event_labels.add('Event')
        ax.axvspan(event_start, event_end, color=color, alpha=0.13, label=label)
        ax.axvline(event_start, color=color, linestyle='--', linewidth=0.85, alpha=0.65)
        ax.axvline(event_end, color=color, linestyle='--', linewidth=0.85, alpha=0.65)

        event_mid = event_start + (event_end - event_start) / 2 + pd.Timedelta(days=event_text_x_offset_days)
        y_text = y_max - (event_text_y_base + event_text_y_step * (event_idx % 3)) * y_range
        event_label = _wrap_text(_event_name(ev), width=event_text_wrap_width)
        ax.text(
            event_mid,
            y_text,
            event_label,
            fontsize=event_text_fontsize,
            color=color,
            ha='center',
            va='top',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor=color, alpha=0.88),
            zorder=7,
        )
        event_summary.append(
            f"{_event_name(ev)} ({ev['start_date']:%Y-%m-%d} to {ev['end_date']:%Y-%m-%d})"
        )

    selected_segment = price_series.loc[row['window_start']:row['window_end']]
    if not selected_segment.empty:
        pred_label = int(row['pred_label'])
        color = PRED_COLORS[pred_label]
        ax.plot(
            selected_segment.index,
            selected_segment.values,
            color=color,
            linewidth=3.8,
            alpha=0.88,
            solid_capstyle='round',
            label=f"Selected predicted window: {pred_label}",
        )

    title = (
        f"{case}: {CONFUSION_CASE_LABELS[case]} | {commodity}\n"
        f"True={int(row['true_label'])}, Pred={int(row['pred_label'])}, "
        f"P(event)={float(row['pred_prob_event']):.3f} | "
        f"{row['window_start']:%Y-%m-%d} to {row['window_end']:%Y-%m-%d}"
    )
    ax.set_title(title, fontsize=title_fontsize)
    ax.set_xlabel('Date', fontsize=axis_label_fontsize)
    if y_axis_label is None:
        y_axis_label = PRICE_YLABEL_BY_COMMODITY.get(commodity, DEFAULT_PRICE_YLABEL)
    ax.set_ylabel(y_axis_label, fontsize=axis_label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_label_fontsize)
    ax.grid(True, color='#e5e7eb', linewidth=0.8)
    ax.spines[['top', 'right']].set_visible(False)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc=legend_loc, fontsize=legend_fontsize, frameon=True)

    return event_summary


def _get_plot_price_range_for_row(row, context_days=20):
    if row is None:
        return None
    price_series = price_df[row['commodity']].dropna()
    plot_start = row['window_start'] - pd.Timedelta(days=context_days)
    plot_end = row['window_end'] + pd.Timedelta(days=context_days)
    visible_prices = price_series.loc[plot_start:plot_end]
    if visible_prices.empty:
        return None
    return float(visible_prices.min()), float(visible_prices.max())


def _compute_common_y_limits(rows, context_days=20, padding_ratio=0.05):
    ranges = [_get_plot_price_range_for_row(row, context_days=context_days) for row in rows if row is not None]
    ranges = [r for r in ranges if r is not None]
    if not ranges:
        return None
    y_min = min(r[0] for r in ranges)
    y_max = max(r[1] for r in ranges)
    padding = (y_max - y_min) * padding_ratio if y_max > y_min else 1.0
    return y_min - padding, y_max + padding


def plot_all_confusion_case_examples(
    sample_n_by_case=None,
    window_key_by_case=None,
    strategy='representative',
    context_days=20,
    figsize=(8.8, 4.9),
    dpi=220,
    common_y_axis=True,
    y_axis_label='Price ($/MMBtu)',
    legend_loc='best',
    legend_fontsize=10,
    event_text_fontsize=10,
    axis_label_fontsize=11,
    tick_label_fontsize=10,
    title_fontsize=11,
    event_text_x_offset_days_by_case=None,
    event_text_y_base_by_case=None,
    event_text_y_step=0.10,
    event_text_wrap_width=34,
    verbose=False,
):
    sample_n_by_case = sample_n_by_case or {}
    window_key_by_case = window_key_by_case or {}
    event_text_x_offset_days_by_case = event_text_x_offset_days_by_case or {}
    event_text_y_base_by_case = event_text_y_base_by_case or {}

    detail_df = build_confusion_example_detail_df()
    rows_by_case = {
        case: select_confusion_example(
            detail_df,
            case,
            strategy=strategy,
            sample_n=sample_n_by_case.get(case),
            window_key=window_key_by_case.get(case),
        )
        for case in ['TP', 'TN', 'FP', 'FN']
    }
    y_limits = (
        _compute_common_y_limits(rows_by_case.values(), context_days=context_days)
        if common_y_axis else None
    )

    results = {}
    for case, row in rows_by_case.items():
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
        event_summary = _plot_single_confusion_example(
            ax,
            row,
            context_days=context_days,
            legend_fontsize=legend_fontsize,
            event_text_fontsize=event_text_fontsize,
            axis_label_fontsize=axis_label_fontsize,
            tick_label_fontsize=tick_label_fontsize,
            title_fontsize=title_fontsize,
            event_text_x_offset_days=event_text_x_offset_days_by_case.get(case, 0),
            event_text_y_base=event_text_y_base_by_case.get(case, 0.10),
            event_text_y_step=event_text_y_step,
            event_text_wrap_width=event_text_wrap_width,
            y_limits=y_limits,
            y_axis_label=y_axis_label,
            legend_loc=legend_loc,
        )

        results[case] = {
            'fig': fig,
            'ax': ax,
            'row': row,
            'event_summary': event_summary,
        }

    if verbose:
        print('Common y-axis limits:', y_limits)
        print('Confusion-case counts:')
        print(detail_df['confusion_case'].value_counts().reindex(['TP', 'TN', 'FP', 'FN'], fill_value=0))

    return results, detail_df, y_limits


# Select the paper examples by stable identifiers rather than test-set order.
CASE_WINDOW_KEYS = {
    'TP': {
        'commodity': 'DHHNGSP',
        'window_start': '2020-11-23',
        'window_end': '2021-03-22',
    },
    'TN': {
        'commodity': 'DHHNGSP',
        'window_start': '2024-07-18',
        'window_end': '2024-11-08',
    },
    'FP': {
        'commodity': 'DHHNGSP',
        'window_start': '2022-08-25',
        'window_end': '2022-12-19',
    },
    'FN': {
        'commodity': 'DHHNGSP',
        'window_start': '2024-12-27',
        'window_end': '2025-04-23',
    },
}

# Optional fallback selectors for cases without a stable window key.
CASE_SAMPLE_N = {case: None for case in ['TP', 'TN', 'FP', 'FN']}

# Adjust these per plot after you inspect the generated figures.
CASE_EVENT_TEXT_X_OFFSET_DAYS = {
    'TP': 28,
    'TN': 0,
    'FP': 0,
    'FN': 37,
}
CASE_EVENT_TEXT_Y_BASE = {
    'TP': 0.10,
    'TN': 0.10,
    'FP': 0.10,
    'FN': 0.10,
}

confusion_case_plot_results, confusion_example_detail_df, shared_y_limits = plot_all_confusion_case_examples(
    sample_n_by_case=CASE_SAMPLE_N,
    window_key_by_case=CASE_WINDOW_KEYS,
    strategy='representative',
    context_days=20,
    common_y_axis=True,
    y_axis_label='Dollars per Million BTU',
    legend_loc='best',
    legend_fontsize=10,
    event_text_fontsize=10,
    axis_label_fontsize=11,
    tick_label_fontsize=10,
    title_fontsize=11,
    event_text_x_offset_days_by_case=CASE_EVENT_TEXT_X_OFFSET_DAYS,
    event_text_y_base_by_case=CASE_EVENT_TEXT_Y_BASE,
    event_text_y_step=0.10,
    event_text_wrap_width=34,
    verbose=False,
)


## Per-commodity benchmark comparison


In [ ]:
# Paper benchmark on the Stage 1 event-safe split.
# Every retained method is fitted independently for each commodity. The table reports
# the equal-weight mean of the six commodity-specific test metrics. The traditional
# one-statistic rules are omitted because they are not part of the reported comparison.


PAPER_BENCHMARK_MODELS = [
    'CNN',
    'ResNet-type CNN',
    'Logistic Regression',
    'MLP 1-layer',
    'MLP 2-layer',
    'Block-sparse MLP',
    'LSTM',
    'FT-Transformer',
]
BENCHMARK_METRIC_COLUMNS = ['accuracy', 'precision', 'recall', 'f1', 'auc']


def metric_summary_row(model_name, y_true_eval, y_pred_eval, y_score_eval=None, **extra):
    row = {
        'model': model_name,
        'accuracy': accuracy_score(y_true_eval, y_pred_eval),
        'precision': precision_score(y_true_eval, y_pred_eval, zero_division=0),
        'recall': recall_score(y_true_eval, y_pred_eval, zero_division=0),
        'f1': f1_score(y_true_eval, y_pred_eval, zero_division=0),
        'auc': (
            roc_auc_score(y_true_eval, y_score_eval)
            if y_score_eval is not None and len(np.unique(y_true_eval)) == 2
            else np.nan
        ),
    }
    row.update(extra)
    return row


def equal_weight_summary(model_name, metrics_df):
    macro_values = metrics_df[BENCHMARK_METRIC_COLUMNS].mean(numeric_only=True)
    return {'model': model_name, **macro_values.to_dict()}


baseline_rows = []
baseline_commodity_metrics = {}


# Reuse the six independently trained CNNs from the preceding binary experiment.
required_cnn_objects = [
    'independent_binary_metrics_df',
    'independent_binary_macro_df',
]
missing_cnn_objects = [name for name in required_cnn_objects if name not in globals()]
if missing_cnn_objects:
    raise RuntimeError(
        'Run the independent per-commodity binary CNN cell before this benchmark. '
        f'Missing objects: {missing_cnn_objects}'
    )

cnn_commodity_metrics_df = independent_binary_metrics_df.copy()
cnn_summary = equal_weight_summary('CNN', cnn_commodity_metrics_df)
expected_cnn_macro = independent_binary_macro_df.loc[
    'macro_avg', BENCHMARK_METRIC_COLUMNS
].astype(float)
if not np.allclose(
    np.asarray([cnn_summary[column] for column in BENCHMARK_METRIC_COLUMNS]),
    expected_cnn_macro.to_numpy(dtype=float),
    equal_nan=True,
):
    raise RuntimeError('The reused CNN row does not match the preceding binary result.')

baseline_rows.append(cnn_summary)
baseline_commodity_metrics['CNN'] = cnn_commodity_metrics_df
print('\n=== CNN row reused from the independent binary experiment ===')
display(pd.DataFrame([cnn_summary]).round(4))


def run_independent_sklearn_benchmark(
    model_name,
    estimator_factory,
    commodities=selected_commodities,
):
    metric_rows = []

    print(f'\n=== Independent {model_name} benchmark ===')
    for commodity_number, commodity in enumerate(commodities, start=1):
        data = get_commodity_binary_splits(commodity)
        estimator = estimator_factory()
        estimator.fit(data['train']['X'], data['train']['y'])
        test_truth = data['test']['y']
        test_prediction = estimator.predict(data['test']['X']).astype(int)
        if hasattr(estimator, 'predict_proba'):
            test_probability = estimator.predict_proba(data['test']['X'])[:, 1]
        elif hasattr(estimator, 'decision_function'):
            test_probability = estimator.decision_function(data['test']['X'])
        else:
            test_probability = test_prediction.astype(float)

        row = metric_summary_row(
            model_name,
            test_truth,
            test_prediction,
            test_probability,
            commodity=str(commodity),
            n_train_before_downsampling=data['n_train_before_downsampling'],
            n_train_fit=len(data['train']['y']),
            n_validation=len(data['validation']['y']),
            n_test=len(test_truth),
        )
        metric_rows.append(row)
        print(
            f'  {commodity_number}/{len(commodities)} {commodity}  '
            f'F1={row["f1"]:.4f}  AUC={row["auc"]:.4f}',
            flush=True,
        )

    metrics_df = pd.DataFrame(metric_rows).reset_index(drop=True)
    return equal_weight_summary(model_name, metrics_df), metrics_df


logistic_params = {'max_iter': 5000, 'class_weight': 'balanced'}
(
    logistic_summary,
    logistic_commodity_metrics_df,
) = run_independent_sklearn_benchmark(
    'Logistic Regression',
    lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=logistic_params['max_iter'],
            class_weight=logistic_params['class_weight'],
            random_state=GLOBAL_SEED,
        )),
    ]),
)
baseline_rows.append(logistic_summary)
baseline_commodity_metrics['Logistic Regression'] = logistic_commodity_metrics_df
display(logistic_commodity_metrics_df)

def train_torch_baseline(
    model,
    train_loader,
    validation_loader,
    device,
    epochs=30,
    lr=1e-3,
    y_train_for_weights=None,
):
    model.to(device)
    model_weights = make_class_weights(y_train_for_weights)
    criterion = nn.CrossEntropyLoss(weight=model_weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_state = None
    best_validation_f1 = -1.0
    best_epoch = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        _, validation_prediction, validation_truth = predict_model(
            model,
            validation_loader,
            device,
        )
        validation_f1 = f1_score(
            validation_truth,
            validation_prediction,
            zero_division=0,
        )
        if validation_f1 > best_validation_f1:
            best_validation_f1 = validation_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    if best_state is None:
        raise RuntimeError('No validation checkpoint was recorded.')
    model.load_state_dict(best_state)
    return model, best_validation_f1, best_epoch


def run_independent_torch_benchmark(
    model_name,
    model_factory,
    epochs=30,
    lr=1e-3,
    seed_offset=0,
    commodities=selected_commodities,
):
    metric_rows = []

    print(f'\n=== Independent {model_name} benchmark ===')
    for commodity_number, commodity in enumerate(commodities, start=1):
        data = get_commodity_binary_splits(commodity)
        model_seed = GLOBAL_SEED + seed_offset
        set_seed(model_seed)
        commodity_model = model_factory(data['train']['X'].shape[1])
        set_seed(model_seed)
        train_loader, validation_loader, test_loader = make_binary_loaders(
            data['train']['X'],
            data['train']['y'],
            data['validation']['X'],
            data['validation']['y'],
            data['test']['X'],
            data['test']['y'],
            batch_size_train=128,
            batch_size_eval=256,
        )
        commodity_model, best_validation_f1, best_epoch = train_torch_baseline(
            commodity_model,
            train_loader,
            validation_loader,
            device,
            epochs=epochs,
            lr=lr,
            y_train_for_weights=data['train']['y'],
        )
        test_logits, test_prediction, test_truth = predict_model(
            commodity_model,
            test_loader,
            device,
        )
        test_probability = torch.softmax(
            torch.tensor(test_logits),
            dim=1,
        )[:, 1].numpy()
        row = metric_summary_row(
            model_name,
            test_truth,
            test_prediction,
            test_probability,
            commodity=str(commodity),
            n_train_before_downsampling=data['n_train_before_downsampling'],
            n_train_fit=len(data['train']['y']),
            n_validation=len(data['validation']['y']),
            n_test=len(test_truth),
            best_validation_f1=best_validation_f1,
            best_epoch=best_epoch,
        )
        metric_rows.append(row)
        commodity_model.to('cpu')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(
            f'  {commodity_number}/{len(commodities)} {commodity}  '
            f'F1={row["f1"]:.4f}  AUC={row["auc"]:.4f}  '
            f'best_epoch={best_epoch}',
            flush=True,
        )

    metrics_df = pd.DataFrame(metric_rows).reset_index(drop=True)
    return equal_weight_summary(model_name, metrics_df), metrics_df


benchmark_torch_specs = [
    {
        'name': 'ResNet-type CNN',
        'factory': lambda input_length: ResNetTypeCNNBinaryClassifier(
            input_length=input_length,
            channels=64,
            n_blocks=4,
            kernel_size=3,
            dropout=0.1,
            num_classes=2,
        ),
        'epochs': 30,
        'lr': 1e-3,
        'seed_offset': 0,
    },
    {
        'name': 'MLP 1-layer',
        'factory': lambda input_length: MLPBinaryClassifier(
            input_length=input_length,
            hidden_layers=1,
            hidden_dim=64,
            dropout=0.3,
            num_classes=2,
        ),
        'epochs': 30,
        'lr': 1e-3,
        'seed_offset': 11,
    },
    {
        'name': 'MLP 2-layer',
        'factory': lambda input_length: MLPBinaryClassifier(
            input_length=input_length,
            hidden_layers=2,
            hidden_dim=64,
            dropout=0.3,
            num_classes=2,
        ),
        'epochs': 30,
        'lr': 1e-3,
        'seed_offset': 12,
    },
    {
        'name': 'Block-sparse MLP',
        'factory': lambda input_length: BlockSparseMLPBinaryClassifier(
            input_length=input_length,
            hidden_dim=128,
            hidden_layers=3,
            n_blocks=4,
            block_bandwidth=0,
            dropout=0.2,
            num_classes=2,
        ),
        'epochs': 30,
        'lr': 1e-3,
        'seed_offset': 28,
    },
    {
        'name': 'LSTM',
        'factory': lambda input_length: LSTMBinaryClassifier(
            input_size=1,
            hidden_dim=64,
            num_layers=1,
            dropout=0.2,
            num_classes=2,
        ),
        'epochs': 30,
        'lr': 1e-3,
        'seed_offset': 30,
    },
    {
        'name': 'FT-Transformer',
        'factory': lambda input_length: FTTransformerBinaryClassifier(
            input_length=input_length,
            d_token=32,
            n_heads=4,
            n_layers=2,
            dropout=0.2,
            num_classes=2,
        ),
        'epochs': 30,
        'lr': 1e-3,
        'seed_offset': 40,
    },
]

for benchmark_spec in benchmark_torch_specs:
    (
        model_summary,
        model_commodity_metrics_df,
    ) = run_independent_torch_benchmark(
        benchmark_spec['name'],
        benchmark_spec['factory'],
        epochs=benchmark_spec['epochs'],
        lr=benchmark_spec['lr'],
        seed_offset=benchmark_spec['seed_offset'],
    )
    baseline_rows.append(model_summary)
    baseline_commodity_metrics[benchmark_spec['name']] = model_commodity_metrics_df
    display(model_commodity_metrics_df)


baseline_comparison_df = pd.DataFrame(baseline_rows).copy()
missing_models = [
    model_name
    for model_name in PAPER_BENCHMARK_MODELS
    if model_name not in set(baseline_comparison_df['model'])
]
if missing_models:
    raise RuntimeError(f'Missing retained benchmark models: {missing_models}')

baseline_comparison_df['model'] = pd.Categorical(
    baseline_comparison_df['model'],
    categories=PAPER_BENCHMARK_MODELS,
    ordered=True,
)
baseline_comparison_df = (
    baseline_comparison_df
    .sort_values('model')
    .reset_index(drop=True)
)
baseline_comparison_df['model'] = baseline_comparison_df['model'].astype(str)
baseline_comparison_df = baseline_comparison_df[
    ['model', *BENCHMARK_METRIC_COLUMNS]
]

print('\n=== Equal-weight mean of independent per-commodity benchmarks ===')
display(baseline_comparison_df.round(4))




In [ ]:
# Verify and export the current per-commodity benchmark components.
# This cell uses the in-memory results from the preceding cell and never loads
# legacy per-commodity CSV files.

required_benchmark_objects = [
    'PAPER_BENCHMARK_MODELS',
    'BENCHMARK_METRIC_COLUMNS',
    'baseline_comparison_df',
    'baseline_commodity_metrics',
]
missing_benchmark_objects = [
    name for name in required_benchmark_objects if name not in globals()
]
if missing_benchmark_objects:
    raise RuntimeError(
        'Run the independent benchmark cell first. '
        f'Missing objects: {missing_benchmark_objects}'
    )

benchmark_component_frames = []
for model_name in PAPER_BENCHMARK_MODELS:
    model_metrics = baseline_commodity_metrics[model_name].copy()
    model_metrics['model'] = model_name
    benchmark_component_frames.append(model_metrics)

components6_benchmark_df = pd.concat(
    benchmark_component_frames,
    ignore_index=True,
)
avg6_benchmark_df = (
    components6_benchmark_df
    .groupby('model', as_index=False)[BENCHMARK_METRIC_COLUMNS]
    .mean()
    .set_index('model')
    .reindex(PAPER_BENCHMARK_MODELS)
    .reset_index()
)
std6_benchmark_df = (
    components6_benchmark_df
    .groupby('model', as_index=False)[BENCHMARK_METRIC_COLUMNS]
    .std(ddof=1)
    .set_index('model')
    .reindex(PAPER_BENCHMARK_MODELS)
    .reset_index()
)

comparison_check = (
    baseline_comparison_df
    .set_index('model')
    .loc[PAPER_BENCHMARK_MODELS, BENCHMARK_METRIC_COLUMNS]
    .to_numpy(dtype=float)
)
if not np.allclose(
    comparison_check,
    avg6_benchmark_df[BENCHMARK_METRIC_COLUMNS].to_numpy(dtype=float),
    equal_nan=True,
):
    raise RuntimeError('The component-level average does not match the paper table.')

print('\n=== Verified benchmark mean across six independently fitted commodity models ===')
display(avg6_benchmark_df.round(4))
print('\n=== Between-commodity standard deviation ===')
display(std6_benchmark_df.round(4))




## Hierarchical event-family classification


In [ ]:
# Full event-detection and three-family hierarchical CNN pipeline
#
# Stage 1 retains the paper's binary split. Stage 2 uses a separate
# class-stratified 60/20/20 split of commodity-event groups. Its test groups
# are restricted to the frozen Stage-1 test pool, so the joint test is unseen by both stages.
#
# Stage 2 training uses each node's ground-truth event-family scope:
#   Node 1: weather vs {geopolitical, supply-policy/macro-financial}
#   Node 2: geopolitical vs supply-policy/macro-financial; true weather is excluded
#
# Testing is genuinely sequential. Only Stage-1 event predictions enter Stage 2, and each
# later node receives only the negative predictions from its parent. All upstream errors
# therefore propagate to the joint four-class result.
# The filtered three-family event table and rolling windows are reused; this cell writes no event files.

required_names = [
    'price_df', 'events_df', 'events_by_commodity', 'build_hierarchical_cnn',
    'TimeSeriesDataset', 'make_binary_loaders', 'make_class_weights', 'FocalLoss',
    'downsample_binary_train_majority',
    'predict_model', 'predict_windows', 'set_seed', 'GLOBAL_SEED',
    'PAPER_EVENT_FILE', 'PAPER_EVENT_FAMILIES', 'PAPER_CUTOFF_DATE',
    'X', 'y', 'meta_df', 'train_idx_bin', 'val_idx_bin', 'test_idx_bin'
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        'Run the import/seed, data-loading, window-builder, and binary-model utility '
        f'cells first. Missing: {missing_names}'
    )

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
CAUSAL_TREE_CATEGORIES = list(PAPER_EVENT_FAMILIES)
CAUSAL_TREE_CATEGORY_MAP = {
    category: label for label, category in enumerate(CAUSAL_TREE_CATEGORIES, start=1)
}
CAUSAL_TREE_LABEL_TO_CATEGORY = {
    label: category for category, label in CAUSAL_TREE_CATEGORY_MAP.items()
}

WEATHER_LABEL = CAUSAL_TREE_CATEGORY_MAP['weather_natural_hazard']
GEOPOLITICAL_LABEL = CAUSAL_TREE_CATEGORY_MAP['geopolitical_security']
SUPPLY_MACRO_LABEL = CAUSAL_TREE_CATEGORY_MAP['supply_policy_macro_financial']

# Reuse the Stage 1 binary split without reassigning any retained window.
# 'none' runs the primary specification. Change to 'train_median_abs' only for
# the prespecified sensitivity run described below.
CAUSAL_TREE_SCALE_MODE = 'none'
CAUSAL_TREE_STAGE1_USE_DOWNSAMPLE = True
CAUSAL_TREE_STAGE1_MAX_NEG_POS_RATIO = 1.5
CAUSAL_TREE_STAGE1_DOWNSAMPLE_SEED = 42
CAUSAL_TREE_STAGE1_THRESHOLD = 0.50
# Full sequential pipeline with commodity-specific event detectors.
CAUSAL_TREE_USE_STAGE1 = True
CAUSAL_TREE_EPOCHS = 30
CAUSAL_TREE_LR = 1e-3
CAUSAL_TREE_BATCH_SIZE_TRAIN = 128
CAUSAL_TREE_BATCH_SIZE_EVAL = 256
CAUSAL_TREE_GAMMA = 2.0
CAUSAL_TREE_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# -----------------------------------------------------------------------------
# Reuse the filtered events and rolling windows already constructed for Stage 1
# -----------------------------------------------------------------------------
required_event_columns = {'category', 'event_structure_category'}
missing_event_columns = required_event_columns.difference(events_df.columns)
if missing_event_columns:
    raise ValueError(
        f'{PAPER_EVENT_FILE.name} is missing columns: {sorted(missing_event_columns)}'
    )
observed_event_families = set(events_df['category'].dropna().unique())
if observed_event_families != set(CAUSAL_TREE_CATEGORIES):
    raise ValueError(
        'Unexpected event families in the paper event table: '
        f'{sorted(observed_event_families)}'
    )

causal_tree_events_df = events_df.copy()
causal_tree_events_by_commodity = events_by_commodity
X_causal_tree = X
meta_causal_tree_df = meta_df.copy()
y_causal_tree = (
    meta_causal_tree_df['category']
    .map(CAUSAL_TREE_CATEGORY_MAP)
    .fillna(0)
    .to_numpy(dtype=int)
)

causal_tree_event_count_df = (
    causal_tree_events_df.groupby('category')
    .agg(
        commodity_event_records=('event_id', 'size'),
        distinct_event_names=('subcategory', 'nunique'),
        commodities=('commodity', 'nunique'),
    )
    .reindex(CAUSAL_TREE_CATEGORIES, fill_value=0)
    .reset_index(names='event_family')
)
print('\n=== Causal event counts after the impact filter ===')
display(causal_tree_event_count_df)

event_window_idx = np.flatnonzero(y_causal_tree != 0)
event_window_labels = y_causal_tree[event_window_idx].astype(int)
expected_labels = set(CAUSAL_TREE_CATEGORY_MAP.values())
if set(np.unique(event_window_labels)) != expected_labels:
    raise ValueError(
        'At least one causal family has no generated event windows. '
        f'Observed labels: {sorted(np.unique(event_window_labels).tolist())}'
    )

causal_tree_window_count_df = (
    pd.Series(event_window_labels)
    .map(CAUSAL_TREE_LABEL_TO_CATEGORY)
    .value_counts()
    .reindex(CAUSAL_TREE_CATEGORIES, fill_value=0)
    .rename_axis('event_family')
    .reset_index(name='event_windows')
)
print('\n=== Generated true-event windows by causal family ===')
display(causal_tree_window_count_df)


# -----------------------------------------------------------------------------
# Reuse of the Stage 1 binary split
# -----------------------------------------------------------------------------
causal_tree_train_idx = np.asarray(train_idx_bin, dtype=int).copy()
causal_tree_val_idx = np.asarray(val_idx_bin, dtype=int).copy()
causal_tree_test_idx = np.asarray(test_idx_bin, dtype=int).copy()
X_stage1 = X
y_stage1_binary = (np.asarray(y) != 0).astype(int)
meta_stage1_df = meta_df.reset_index(drop=True)
causal_tree_commodities = pd.Index(
    meta_stage1_df['commodity']
).drop_duplicates().tolist()

full_split_assignment = pd.Series(index=meta_causal_tree_df.index, dtype=object)
full_split_assignment.loc[causal_tree_train_idx] = 'train'
full_split_assignment.loc[causal_tree_val_idx] = 'validation'
full_split_assignment.loc[causal_tree_test_idx] = 'test'

kept_event_meta = meta_causal_tree_df.loc[event_window_idx].copy()
kept_event_meta['sample_index'] = event_window_idx
kept_event_meta['label'] = event_window_labels
kept_event_meta['split'] = full_split_assignment.loc[event_window_idx].to_numpy()
kept_event_meta['event_group'] = (
    kept_event_meta['commodity'].astype(str)
    + '::'
    + kept_event_meta['event_id'].astype(str)
)
purged_event_windows = int(kept_event_meta['split'].isna().sum())
kept_event_meta = kept_event_meta.dropna(subset=['split']).copy()

if kept_event_meta.groupby('event_group')['split'].nunique().max() != 1:
    leaked_groups = kept_event_meta.groupby('event_group')['split'].nunique()
    leaked_groups = leaked_groups[leaked_groups > 1]
    raise AssertionError(
        f'Commodity-event groups cross subsets: {leaked_groups.index.tolist()}'
    )
if kept_event_meta.groupby('event_group')['label'].nunique().max() != 1:
    raise AssertionError('A commodity-event group has multiple causal-family labels.')

causal_tree_event_groups_df = (
    kept_event_meta.groupby('event_group', as_index=False)
    .agg(
        commodity=('commodity', 'first'),
        event_id=('event_id', 'first'),
        label=('label', 'first'),
        split=('split', 'first'),
        windows=('sample_index', 'size'),
    )
)

expected_full_labels = {0, *expected_labels}
for split_name, split_idx in [
    ('train', causal_tree_train_idx),
    ('validation', causal_tree_val_idx),
    ('test', causal_tree_test_idx),
]:
    observed_labels = set(y_causal_tree[split_idx].astype(int))
    if observed_labels != expected_full_labels:
        raise ValueError(
            f'{split_name} does not contain every event family: {sorted(observed_labels)}'
        )

raw_observation_owner = {}
raw_observation_conflicts = []
for split_name, split_idx in [
    ('train', causal_tree_train_idx),
    ('validation', causal_tree_val_idx),
    ('test', causal_tree_test_idx),
]:
    for row in meta_causal_tree_df.loc[split_idx].itertuples():
        for position in range(
            int(row.series_start_position), int(row.series_end_position) + 1
        ):
            observation_key = (str(row.commodity), position)
            previous_owner = raw_observation_owner.setdefault(observation_key, split_name)
            if previous_owner != split_name:
                raw_observation_conflicts.append((observation_key, previous_owner, split_name))
if raw_observation_conflicts:
    raise AssertionError(
        f'Raw price observations cross subsets: {len(raw_observation_conflicts)} conflicts.'
    )

causal_tree_commodity_splits = {}
for commodity in causal_tree_commodities:
    causal_tree_commodity_splits[commodity] = {}
    for split_key, split_idx in [
        ('train_idx', causal_tree_train_idx),
        ('val_idx', causal_tree_val_idx),
        ('test_idx', causal_tree_test_idx),
    ]:
        commodity_mask = (
            meta_stage1_df.loc[split_idx, 'commodity'].astype(str).to_numpy()
            == str(commodity)
        )
        causal_tree_commodity_splits[commodity][split_key] = split_idx[commodity_mask]

print(
    f'Frozen Stage-1 split and leakage audit retained; '
    f'{purged_event_windows} event windows were removed by its boundary purge.'
)


# -----------------------------------------------------------------------------
# Stage-2-only class-stratified split; Stage 1 above remains frozen
# -----------------------------------------------------------------------------
causal_stage1_test_idx = causal_tree_test_idx.copy()
kept_event_meta = kept_event_meta.rename(columns={'split': 'stage1_split'})
stage2_group_pool_df = causal_tree_event_groups_df.rename(
    columns={'split': 'stage1_split'}
).copy()

CAUSAL_TREE_STAGE2_TRAIN_RATIO = 0.60
CAUSAL_TREE_STAGE2_VAL_RATIO = 0.20
CAUSAL_TREE_STAGE2_TEST_RATIO = 0.20
CAUSAL_TREE_STAGE2_SPLIT_SEED = GLOBAL_SEED + 2026
if not np.isclose(
    CAUSAL_TREE_STAGE2_TRAIN_RATIO
    + CAUSAL_TREE_STAGE2_VAL_RATIO
    + CAUSAL_TREE_STAGE2_TEST_RATIO,
    1.0,
):
    raise ValueError('Stage-2 split ratios must sum to one.')


def choose_stage2_group_subset(group_df, subset_size, target_windows, seed):
    group_df = group_df.sort_values('event_group').reset_index(drop=True)
    group_ids = group_df['event_group'].tolist()
    window_lookup = group_df.set_index('event_group')['windows'].to_dict()
    if subset_size <= 0:
        return []
    if len(group_ids) < subset_size:
        raise ValueError('Not enough eligible event groups for this Stage-2 subset.')
    if len(group_ids) == subset_size:
        return group_ids
    combination_count = math.comb(len(group_ids), subset_size)
    if combination_count <= 200000:
        candidates = itertools.combinations(group_ids, subset_size)
    else:
        rng = np.random.default_rng(seed)
        candidates = (
            tuple(sorted(rng.choice(
                group_ids, size=subset_size, replace=False
            ).tolist()))
            for _ in range(50000)
        )
    best_choice = None
    best_score = None
    for choice in candidates:
        selected_windows = sum(window_lookup[group_id] for group_id in choice)
        score = (abs(selected_windows - target_windows), tuple(choice))
        if best_score is None or score < best_score:
            best_score = score
            best_choice = list(choice)
    return best_choice


stage2_group_assignment = {}
for label, label_groups in stage2_group_pool_df.groupby('label', sort=True):
    label_groups = label_groups.copy()
    group_n = len(label_groups)
    test_n = max(1, int(round(group_n * CAUSAL_TREE_STAGE2_TEST_RATIO)))
    val_n = max(1, int(round(group_n * CAUSAL_TREE_STAGE2_VAL_RATIO)))
    while group_n - test_n - val_n < 1:
        if test_n >= val_n and test_n > 1:
            test_n -= 1
        elif val_n > 1:
            val_n -= 1
        else:
            raise ValueError(f'Class {label} has too few groups for three subsets.')
    total_windows = int(label_groups['windows'].sum())
    test_candidates = label_groups.loc[label_groups['stage1_split'].eq('test')]
    if len(test_candidates) < test_n:
        raise ValueError(
            f'Class {label} has only {len(test_candidates)} Stage-1 test groups; '
            f'{test_n} are required for leakage-free Stage-2 testing.'
        )
    test_groups = choose_stage2_group_subset(
        test_candidates, test_n,
        total_windows * CAUSAL_TREE_STAGE2_TEST_RATIO,
        CAUSAL_TREE_STAGE2_SPLIT_SEED + 100 * int(label) + 1,
    )
    remaining = label_groups.loc[
        ~label_groups['event_group'].isin(test_groups)
    ].copy()
    validation_candidates = remaining.loc[
        ~remaining['stage1_split'].eq('test')
    ]
    if len(validation_candidates) < val_n:
        validation_candidates = remaining
    val_groups = choose_stage2_group_subset(
        validation_candidates, val_n,
        total_windows * CAUSAL_TREE_STAGE2_VAL_RATIO,
        CAUSAL_TREE_STAGE2_SPLIT_SEED + 100 * int(label) + 2,
    )
    train_groups = remaining.loc[
        ~remaining['event_group'].isin(val_groups), 'event_group'
    ].tolist()
    stage2_group_assignment.update({group_id: 'train' for group_id in train_groups})
    stage2_group_assignment.update({group_id: 'validation' for group_id in val_groups})
    stage2_group_assignment.update({group_id: 'test' for group_id in test_groups})

kept_event_meta['stage2_split'] = kept_event_meta['event_group'].map(
    stage2_group_assignment
)
if kept_event_meta['stage2_split'].isna().any():
    raise AssertionError('At least one Stage-2 event group was not assigned.')

# Protect test first and validation second. Lower-priority windows are removed
# if reassignment makes their raw observations overlap a protected subset.
stage2_keep_mask = pd.Series(True, index=kept_event_meta.index)
for commodity, commodity_rows in kept_event_meta.groupby('commodity', sort=False):
    for protected_split, lower_splits in [
        ('test', {'train', 'validation'}),
        ('validation', {'train'}),
    ]:
        protected_rows = commodity_rows.loc[
            stage2_keep_mask.loc[commodity_rows.index]
            & commodity_rows['stage2_split'].eq(protected_split)
        ]
        protected_positions = set()
        for row in protected_rows.itertuples():
            protected_positions.update(range(
                int(row.series_start_position), int(row.series_end_position) + 1
            ))
        lower_rows = commodity_rows.loc[
            stage2_keep_mask.loc[commodity_rows.index]
            & commodity_rows['stage2_split'].isin(lower_splits)
        ]
        for row in lower_rows.itertuples():
            if any(
                position in protected_positions
                for position in range(
                    int(row.series_start_position), int(row.series_end_position) + 1
                )
            ):
                stage2_keep_mask.loc[row.Index] = False

stage2_purged_event_windows = int((~stage2_keep_mask).sum())
stage2_event_meta = kept_event_meta.loc[stage2_keep_mask].copy()
for commodity, commodity_rows in stage2_event_meta.groupby('commodity', sort=False):
    observation_owner = {}
    for row in commodity_rows.itertuples():
        for position in range(
            int(row.series_start_position), int(row.series_end_position) + 1
        ):
            previous_owner = observation_owner.setdefault(position, row.stage2_split)
            if previous_owner != row.stage2_split:
                raise AssertionError(
                    f'Stage-2 raw observations cross subsets for {commodity}.'
                )

causal_tree_train_idx = stage2_event_meta.loc[
    stage2_event_meta['stage2_split'].eq('train'), 'sample_index'
].to_numpy(dtype=int)
causal_tree_val_idx = stage2_event_meta.loc[
    stage2_event_meta['stage2_split'].eq('validation'), 'sample_index'
].to_numpy(dtype=int)
causal_tree_test_idx = stage2_event_meta.loc[
    stage2_event_meta['stage2_split'].eq('test'), 'sample_index'
].to_numpy(dtype=int)
if not set(causal_tree_test_idx).issubset(set(causal_stage1_test_idx)):
    raise AssertionError('Every Stage-2 test event must be in the Stage-1 test set.')

causal_tree_event_groups_df = (
    stage2_event_meta.groupby('event_group', as_index=False)
    .agg(
        commodity=('commodity', 'first'),
        event_id=('event_id', 'first'),
        label=('label', 'first'),
        split=('stage2_split', 'first'),
        windows=('sample_index', 'size'),
        stage1_split=('stage1_split', 'first'),
    )
)
stage2_split_rows = []
for split_name, split_idx in [
    ('train', causal_tree_train_idx),
    ('validation', causal_tree_val_idx),
    ('test', causal_tree_test_idx),
]:
    observed_labels = set(y_causal_tree[split_idx].astype(int))
    if observed_labels != expected_labels:
        raise ValueError(
            f'Stage-2 {split_name} lacks a class: {sorted(observed_labels)}'
        )
    split_groups = causal_tree_event_groups_df.loc[
        causal_tree_event_groups_df['split'].eq(split_name)
    ]
    group_counts = split_groups['label'].value_counts().to_dict()
    window_counts = pd.Series(y_causal_tree[split_idx]).value_counts().to_dict()
    for label, class_name in CAUSAL_TREE_LABEL_TO_CATEGORY.items():
        stage2_split_rows.append({
            'split': split_name,
            'class_name': class_name,
            'commodity_event_groups': int(group_counts.get(label, 0)),
            'windows': int(window_counts.get(label, 0)),
        })
causal_tree_split_counts_df = pd.DataFrame(stage2_split_rows)
print('\n=== Stage 1 remains unchanged; Stage 2 uses a stratified split ===')
display(causal_tree_split_counts_df.pivot(
    index='class_name', columns='split',
    values=['commodity_event_groups', 'windows']
))
print(f'Stage-2 reassignment purge removed {stage2_purged_event_windows} windows.')
print('Stage-2 event-group and raw-observation leakage audits passed.')


# -----------------------------------------------------------------------------
# Optional commodity-level scaling estimated from training observations only
# -----------------------------------------------------------------------------
def scale_causal_windows_by_commodity(X_values, metadata, training_idx, mode):
    allowed_modes = {'none', 'train_median_abs'}
    if mode not in allowed_modes:
        raise ValueError(f'CAUSAL_TREE_SCALE_MODE must be one of {sorted(allowed_modes)}.')
    scaled_values = np.asarray(X_values, dtype=np.float32).copy()
    scale_rows = []
    for commodity in causal_tree_commodities:
        commodity_mask = metadata['commodity'].astype(str).eq(str(commodity)).to_numpy()
        commodity_all_idx = np.flatnonzero(commodity_mask)
        commodity_train_idx = np.intersect1d(
            training_idx, commodity_all_idx, assume_unique=False
        )
        commodity_series = price_df[commodity].dropna().sort_index()
        covered_positions = np.zeros(len(commodity_series), dtype=bool)
        for row in metadata.loc[commodity_train_idx].itertuples():
            covered_positions[
                int(row.series_start_position):int(row.series_end_position) + 1
            ] = True
        training_values = commodity_series.iloc[covered_positions].to_numpy(dtype=float)
        if len(training_values) == 0:
            raise ValueError(f'No training observations are available for {commodity}.')
        if mode == 'none':
            scale = 1.0
        else:
            scale = float(np.median(np.abs(training_values)))
            if not np.isfinite(scale) or scale <= 1e-8:
                scale = float(np.mean(np.abs(training_values)))
            if not np.isfinite(scale) or scale <= 1e-8:
                scale = 1.0
        scaled_values[commodity_all_idx] /= scale
        scale_rows.append({
            'commodity': commodity,
            'mode': mode,
            'training_unique_observations': int(covered_positions.sum()),
            'positive_scale': scale,
        })
    return scaled_values, pd.DataFrame(scale_rows)


X_causal_tree_model, causal_tree_scale_df = scale_causal_windows_by_commodity(
    X_causal_tree, meta_causal_tree_df, causal_tree_train_idx, CAUSAL_TREE_SCALE_MODE
)
print(f'\n=== Commodity scaling mode: {CAUSAL_TREE_SCALE_MODE} ===')
display(causal_tree_scale_df)


# -----------------------------------------------------------------------------
# Stage 1: one independently trained event/no-event CNN per commodity
# -----------------------------------------------------------------------------
def train_causal_stage1_model(model, train_loader, val_loader, criterion, optimizer):
    model.to(CAUSAL_TREE_DEVICE)
    best_val_f1 = -1.0
    best_epoch = 0
    for epoch in range(1, CAUSAL_TREE_EPOCHS + 1):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(CAUSAL_TREE_DEVICE)
            y_batch = y_batch.to(CAUSAL_TREE_DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_pred_parts = []
        val_true_parts = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                logits = model(X_batch.to(CAUSAL_TREE_DEVICE))
                val_pred_parts.append(torch.argmax(logits, dim=1).cpu().numpy())
                val_true_parts.append(y_batch.numpy())
        val_pred = np.concatenate(val_pred_parts)
        val_true = np.concatenate(val_true_parts)
        val_f1 = f1_score(val_true, val_pred, zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = float(val_f1)
            best_epoch = epoch

    # Use the reported Stage 1 rule based on the last epoch and argmax.
    return model, best_val_f1, best_epoch


def train_causal_stage1_detectors():
    metric_rows = []
    prediction_frames = []
    models = {}

    print('\n=== Stage 1: commodity-specific event/no-event CNN detectors ===')
    for commodity in causal_tree_commodities:
        split = causal_tree_commodity_splits[commodity]
        train_idx = split['train_idx']
        val_idx = split['val_idx']
        test_idx = split['test_idx']

        X_train = X_stage1[train_idx]
        y_train = y_stage1_binary[train_idx]
        X_val = X_stage1[val_idx]
        y_val = y_stage1_binary[val_idx]
        X_test = X_stage1[test_idx]
        y_test = y_stage1_binary[test_idx]

        X_train_fit = X_train.copy()
        y_train_fit = y_train.copy()
        if CAUSAL_TREE_STAGE1_USE_DOWNSAMPLE:
            X_train_fit, y_train_fit = downsample_binary_train_majority(
                X_train_fit, y_train_fit,
                target_neg_pos_ratio=CAUSAL_TREE_STAGE1_MAX_NEG_POS_RATIO,
                random_seed=CAUSAL_TREE_STAGE1_DOWNSAMPLE_SEED,
            )

        set_seed(GLOBAL_SEED)
        train_loader, val_loader, test_loader = make_binary_loaders(
            X_train_fit,
            y_train_fit,
            X_val,
            y_val,
            X_test,
            y_test,
            batch_size_train=CAUSAL_TREE_BATCH_SIZE_TRAIN,
            batch_size_eval=CAUSAL_TREE_BATCH_SIZE_EVAL,
        )
        class_weights = make_class_weights(y_train_fit)
        model = build_hierarchical_cnn(
            input_length=X_train_fit.shape[1], num_classes=2, dropout=0.3
        )
        criterion = FocalLoss(
            alpha=class_weights.to(CAUSAL_TREE_DEVICE), gamma=CAUSAL_TREE_GAMMA
        )
        optimizer = torch.optim.Adam(model.parameters(), lr=CAUSAL_TREE_LR)
        model, best_val_f1, best_epoch = train_causal_stage1_model(
            model, train_loader, val_loader, criterion, optimizer
        )

        test_logits, test_pred, test_true = predict_model(
            model, test_loader, CAUSAL_TREE_DEVICE
        )
        test_probability = torch.softmax(
            torch.tensor(test_logits), dim=1
        )[:, 1].numpy()
        row = {
            'commodity': commodity,
            'n_train_fit': len(y_train_fit),
            'train_no_event': int((y_train_fit == 0).sum()),
            'train_event': int((y_train_fit == 1).sum()),
            'threshold': CAUSAL_TREE_STAGE1_THRESHOLD,
            'checkpoint': 'final_epoch',
            'diagnostic_best_epoch': best_epoch,
            'diagnostic_best_val_f1': best_val_f1,
        }
        row.update(binary_metric_values(test_true, test_pred, test_probability))
        metric_rows.append(row)
        prediction_frames.append(pd.DataFrame({
            'commodity': commodity,
            'sample_index': test_idx,
            'true_event_binary': test_true,
            'pred_event_binary': test_pred,
            'event_probability': test_probability,
        }))
        models[commodity] = model
        print(
            f'  {commodity}: f1={row["f1"]:.4f}, precision={row["precision"]:.4f}, '
            f'recall={row["recall"]:.4f}, auc={row["roc_auc"]:.4f}',
            flush=True,
        )

    metrics_df = pd.DataFrame(metric_rows)
    macro_columns = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
    macro_row = {'scope': 'six-commodity unweighted mean'}
    macro_row.update(metrics_df[macro_columns].mean().to_dict())
    macro_df = pd.DataFrame([macro_row])

    prediction_df = pd.concat(prediction_frames, ignore_index=True)
    true_all = prediction_df['true_event_binary'].to_numpy(dtype=int)
    pred_all = prediction_df['pred_event_binary'].to_numpy(dtype=int)
    prob_all = prediction_df['event_probability'].to_numpy(dtype=float)
    pooled_row = {'scope': 'pooled commodity-specific Stage 1'}
    pooled_row.update(binary_metric_values(true_all, pred_all, prob_all))
    pooled_row['n_test'] = len(true_all)
    pooled_df = pd.DataFrame([pooled_row])
    confusion_df = pd.DataFrame(
        confusion_matrix(true_all, pred_all, labels=[0, 1]),
        index=['true_no_event', 'true_event'],
        columns=['pred_no_event', 'pred_event'],
    )
    return (
        metrics_df,
        prediction_df,
        models,
        pooled_df,
        macro_df,
        confusion_df,
    )


# -----------------------------------------------------------------------------
# Event- and class-balanced CNN node training
# -----------------------------------------------------------------------------
class CausalTreeWeightedDataset(torch.utils.data.Dataset):
    def __init__(self, X_values, y_values, sample_weights):
        self.X = torch.tensor(X_values[:, None, :], dtype=torch.float32)
        self.y = torch.tensor(y_values, dtype=torch.long)
        self.sample_weights = torch.tensor(sample_weights, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index], self.sample_weights[index]


def make_event_and_class_balanced_weights(binary_labels, event_group_ids):
    """Give each commodity-event record equal total mass, then balance classes."""
    weight_df = pd.DataFrame({
        'label': np.asarray(binary_labels, dtype=int),
        'event_group': pd.Series(event_group_ids).astype(str).to_numpy(),
    })
    windows_per_event = weight_df.groupby('event_group')['event_group'].transform('size').astype(float)
    weight_df['base_weight'] = 1.0 / windows_per_event
    class_mass = weight_df.groupby('label')['base_weight'].sum().reindex([0, 1], fill_value=0.0)
    if (class_mass <= 0).any():
        raise ValueError(f'Both node classes are required. Effective event mass: {class_mass.to_dict()}')
    class_factor = class_mass.sum() / (2.0 * class_mass)
    weights = weight_df['base_weight'] * weight_df['label'].map(class_factor)
    weights = weights / weights.mean()
    return weights.to_numpy(dtype=np.float32)


def weighted_focal_loss(logits, targets, sample_weights, gamma=2.0):
    log_prob = F.log_softmax(logits, dim=1)
    prob = log_prob.exp()
    row_idx = torch.arange(targets.shape[0], device=targets.device)
    log_p_t = log_prob[row_idx, targets]
    p_t = prob[row_idx, targets]
    sample_loss = -((1.0 - p_t) ** gamma) * log_p_t
    return (sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)


def causal_tree_predict(model, sample_idx):
    sample_idx = np.asarray(sample_idx, dtype=int)
    return predict_windows(
        model,
        X_causal_tree_model[sample_idx],
        CAUSAL_TREE_DEVICE,
        batch_size=CAUSAL_TREE_BATCH_SIZE_EVAL,
    )


def filter_scope(sample_idx, scope_labels):
    sample_idx = np.asarray(sample_idx, dtype=int)
    return sample_idx[np.isin(y_causal_tree[sample_idx], scope_labels)]


def train_causal_tree_node(node_name, positive_label, scope_labels, node_seed):
    train_idx = filter_scope(causal_tree_train_idx, scope_labels)
    val_idx = filter_scope(causal_tree_val_idx, scope_labels)
    y_train = (y_causal_tree[train_idx] == positive_label).astype(int)
    y_val = (y_causal_tree[val_idx] == positive_label).astype(int)
    train_event_groups = (
        meta_causal_tree_df.loc[train_idx, 'commodity'].astype(str).to_numpy()
        + '::'
        + meta_causal_tree_df.loc[train_idx, 'event_id'].astype(str).to_numpy()
    )
    train_weights = make_event_and_class_balanced_weights(y_train, train_event_groups)

    set_seed(node_seed)
    train_ds = CausalTreeWeightedDataset(
        X_causal_tree_model[train_idx], y_train, train_weights
    )
    loader_generator = torch.Generator().manual_seed(node_seed)
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=CAUSAL_TREE_BATCH_SIZE_TRAIN,
        shuffle=True,
        generator=loader_generator,
        num_workers=0,
    )
    model = build_hierarchical_cnn(
        input_length=X_causal_tree_model.shape[1], num_classes=2, dropout=0.3
    ).to(CAUSAL_TREE_DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CAUSAL_TREE_LR)
    best_state = copy.deepcopy(model.state_dict())
    best_val_f1 = -1.0
    best_epoch = 0

    for epoch in range(1, CAUSAL_TREE_EPOCHS + 1):
        model.train()
        for X_batch, y_batch, sample_weight_batch in train_loader:
            X_batch = X_batch.to(CAUSAL_TREE_DEVICE)
            y_batch = y_batch.to(CAUSAL_TREE_DEVICE)
            sample_weight_batch = sample_weight_batch.to(CAUSAL_TREE_DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = weighted_focal_loss(
                logits, y_batch, sample_weight_batch, gamma=CAUSAL_TREE_GAMMA
            )
            loss.backward()
            optimizer.step()

        val_pred, _ = causal_tree_predict(model, val_idx)
        val_f1 = f1_score(y_val, val_pred, zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = float(val_f1)
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return {
        'node_name': node_name,
        'model': model,
        'positive_label': positive_label,
        'scope_labels': tuple(scope_labels),
        'train_idx': train_idx,
        'val_idx': val_idx,
        'best_epoch': best_epoch,
        'best_val_f1': best_val_f1,
    }


def binary_metric_values(y_true, y_pred, probability):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return {
            'accuracy': np.nan,
            'balanced_accuracy': np.nan,
            'precision': np.nan,
            'recall': np.nan,
            'f1': np.nan,
            'roc_auc': np.nan,
            'pr_auc': np.nan,
        }
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability),
        'pr_auc': average_precision_score(y_true, probability),
    }


# -----------------------------------------------------------------------------
# Optionally train Stage 1: one event/no-event CNN per commodity
# -----------------------------------------------------------------------------
if CAUSAL_TREE_USE_STAGE1:
    (
        causal_stage1_metrics_df,
        causal_stage1_prediction_df,
        causal_stage1_models,
        causal_stage1_pooled_df,
        causal_stage1_macro_df,
        causal_stage1_confusion_df,
    ) = train_causal_stage1_detectors()
else:
    causal_stage1_metrics_df = None
    causal_stage1_prediction_df = None
    causal_stage1_models = {}
    causal_stage1_pooled_df = None
    causal_stage1_macro_df = None
    causal_stage1_confusion_df = None
    print(
        '\nStage 1 routing is not included in this event-only test. '
        'Stage 2 is evaluated on ground-truth event windows.'
    )

# -----------------------------------------------------------------------------
# Train two CNN nodes on their ground-truth scopes
# -----------------------------------------------------------------------------
node_specs = [
    {
        'node_name': 'Node 1: weather vs rest',
        'positive_label': WEATHER_LABEL,
        'scope_labels': [WEATHER_LABEL, GEOPOLITICAL_LABEL, SUPPLY_MACRO_LABEL],
    },
    {
        'node_name': 'Node 2: geopolitical vs supply-policy/macro-financial',
        'positive_label': GEOPOLITICAL_LABEL,
        'scope_labels': [GEOPOLITICAL_LABEL, SUPPLY_MACRO_LABEL],
    },
]

causal_tree_nodes = {}
print(f'\nTraining causal decision tree on device: {CAUSAL_TREE_DEVICE}')
for node_number, spec in enumerate(node_specs, start=1):
    print(f'Training {spec["node_name"]}...', flush=True)
    trained_node = train_causal_tree_node(
        spec['node_name'],
        spec['positive_label'],
        spec['scope_labels'],
        GLOBAL_SEED + 100 * spec['positive_label'],
    )
    causal_tree_nodes[node_number] = trained_node
    print(
        f'  best_epoch={trained_node["best_epoch"]}, '
        f'best_val_f1={trained_node["best_val_f1"]:.4f}',
        flush=True,
    )


# -----------------------------------------------------------------------------
# Sequential inference on a joint test set unseen by both stages
# -----------------------------------------------------------------------------
if CAUSAL_TREE_USE_STAGE1:
    causal_stage1_prediction_df = causal_stage1_prediction_df.sort_values(
        'sample_index'
    ).reset_index(drop=True)

    # Stage 2 may train on event groups from the old Stage-1 test pool.
    # Exclude those event windows and nearby no-event windows from the final test.
    non_test_event_meta = stage2_event_meta.loc[
        stage2_event_meta['stage2_split'].isin(['train', 'validation'])
    ]
    non_test_positions_by_commodity = {}
    for commodity, rows in non_test_event_meta.groupby('commodity', sort=False):
        occupied = set()
        for row in rows.itertuples():
            occupied.update(range(
                int(row.series_start_position), int(row.series_end_position) + 1
            ))
        non_test_positions_by_commodity[str(commodity)] = occupied

    stage2_test_event_set = set(causal_tree_test_idx.tolist())
    joint_test_indices = []
    dropped_joint_no_event_windows = 0
    for row in meta_stage1_df.loc[causal_stage1_test_idx].itertuples():
        sample_index = int(row.Index)
        if y_stage1_binary[sample_index] == 1:
            if sample_index in stage2_test_event_set:
                joint_test_indices.append(sample_index)
            continue
        occupied = non_test_positions_by_commodity.get(str(row.commodity), set())
        has_overlap = any(
            position in occupied
            for position in range(
                int(row.series_start_position), int(row.series_end_position) + 1
            )
        )
        if has_overlap:
            dropped_joint_no_event_windows += 1
        else:
            joint_test_indices.append(sample_index)

    joint_test_set = set(joint_test_indices)
    joint_prediction_df = causal_stage1_prediction_df.loc[
        causal_stage1_prediction_df['sample_index'].isin(joint_test_set)
    ].sort_values('sample_index').reset_index(drop=True)
    if set(causal_tree_test_idx.tolist()).difference(
        joint_prediction_df['sample_index']
    ):
        raise AssertionError(
            'A Stage-2 test event lacks an out-of-sample Stage-1 prediction.'
        )

    test_idx_all = joint_prediction_df['sample_index'].to_numpy(dtype=int)
    stage1_pred_binary = joint_prediction_df[
        'pred_event_binary'
    ].to_numpy(dtype=int)
    stage1_event_probability = joint_prediction_df[
        'event_probability'
    ].to_numpy(dtype=float)
    # A Stage-1 negative receives the final no-event label immediately.
    final_pred_labels = np.zeros(len(test_idx_all), dtype=int)
    stage2_positions = np.flatnonzero(stage1_pred_binary == 1)
    print(
        f'Final joint test retained {len(test_idx_all)} windows; '
        f'removed {dropped_joint_no_event_windows} no-event windows that overlap '
        'Stage-2 train/validation event windows.'
    )
else:
    test_idx_all = np.asarray(causal_tree_test_idx, dtype=int)
    stage1_pred_binary = np.ones(len(test_idx_all), dtype=int)
    stage1_event_probability = np.full(len(test_idx_all), np.nan)
    final_pred_labels = np.zeros(len(test_idx_all), dtype=int)
    stage2_positions = np.arange(len(test_idx_all), dtype=int)

test_true_labels = y_causal_tree[test_idx_all].astype(int)
stage2_idx = test_idx_all[stage2_positions]

node1_pred, node1_prob = causal_tree_predict(causal_tree_nodes[1]['model'], stage2_idx)
node1_positive_mask = node1_pred == 1
final_pred_labels[stage2_positions[node1_positive_mask]] = WEATHER_LABEL

node2_parent_positions = stage2_positions[~node1_positive_mask]
node2_idx = test_idx_all[node2_parent_positions]
node2_pred, node2_prob = causal_tree_predict(causal_tree_nodes[2]['model'], node2_idx)
node2_positive_mask = node2_pred == 1
final_pred_labels[node2_parent_positions[node2_positive_mask]] = GEOPOLITICAL_LABEL
final_pred_labels[node2_parent_positions[~node2_positive_mask]] = SUPPLY_MACRO_LABEL


def routed_node_result(node_number, routed_idx, routed_pred, routed_prob):
    node = causal_tree_nodes[node_number]
    routed_idx = np.asarray(routed_idx, dtype=int)
    true_multiclass = y_causal_tree[routed_idx].astype(int)
    in_scope_mask = np.isin(true_multiclass, node['scope_labels'])
    in_scope_true = (true_multiclass[in_scope_mask] == node['positive_label']).astype(int)
    in_scope_pred = routed_pred[in_scope_mask]
    in_scope_prob = routed_prob[in_scope_mask]
    routed_n = len(routed_idx)
    in_scope_n = int(in_scope_mask.sum())
    out_of_scope_n = routed_n - in_scope_n
    correct_in_scope_n = int((in_scope_pred == in_scope_true).sum())
    # Out-of-scope arrivals have no valid local binary target and are therefore
    # counted as path errors in effective_routed_accuracy.
    row = {
        'node': node['node_name'],
        'positive_family': CAUSAL_TREE_LABEL_TO_CATEGORY[node['positive_label']],
        'routed_test_n': routed_n,
        'in_scope_test_n': in_scope_n,
        'upstream_out_of_scope_n': out_of_scope_n,
        'route_purity': in_scope_n / routed_n if routed_n else np.nan,
        'correct_in_scope_n': correct_in_scope_n,
        'effective_routed_error_n': routed_n - correct_in_scope_n,
        'effective_routed_accuracy': (
            correct_in_scope_n / routed_n if routed_n else np.nan
        ),
        'best_epoch': node['best_epoch'],
        'best_val_f1': node['best_val_f1'],
    }
    conditional_metrics = binary_metric_values(
        in_scope_true, in_scope_pred, in_scope_prob
    )
    row.update({
        f'conditional_{metric_name}': metric_value
        for metric_name, metric_value in conditional_metrics.items()
    })
    return row


causal_tree_node_metrics_df = pd.DataFrame([
    routed_node_result(1, stage2_idx, node1_pred, node1_prob),
    routed_node_result(2, node2_idx, node2_pred, node2_prob),
])

route_rows = []
if CAUSAL_TREE_USE_STAGE1:
    route_rows.append({
        'route_stage': 'Stage 1: event vs no event',
        'n_windows': len(test_idx_all),
        'assigned_positive': len(stage2_idx),
        'passed_to_next_node': len(stage2_idx),
    })
route_rows.extend([
    {
        'route_stage': 'Stage 2 Node 1: weather vs rest',
        'n_windows': len(stage2_idx),
        'assigned_positive': int(node1_positive_mask.sum()),
        'passed_to_next_node': len(node2_idx),
    },
    {
        'route_stage': 'Stage 2 Node 2: geopolitical vs supply-policy/macro-financial',
        'n_windows': len(node2_idx),
        'assigned_positive': int(node2_positive_mask.sum()),
        'passed_to_next_node': 0,
    },
])
route_summary_df = pd.DataFrame(route_rows)

if CAUSAL_TREE_USE_STAGE1:
    final_label_to_class = {0: 'no_event', **CAUSAL_TREE_LABEL_TO_CATEGORY}
else:
    final_label_to_class = dict(CAUSAL_TREE_LABEL_TO_CATEGORY)
final_labels = list(final_label_to_class)
final_precision, final_recall, final_f1, final_support = precision_recall_fscore_support(
    test_true_labels,
    final_pred_labels,
    labels=final_labels,
    zero_division=0,
)
causal_tree_per_class_df = pd.DataFrame({
    'class_label': final_labels,
    'class_name': [final_label_to_class[label] for label in final_labels],
    'precision': final_precision,
    'recall': final_recall,
    'f1': final_f1,
    'support': final_support.astype(int),
})

causal_tree_final_metrics_df = pd.DataFrame([{
    'model': (
        'Commodity-specific Stage 1 + hierarchical causal CNN tree'
        if CAUSAL_TREE_USE_STAGE1 else 'Event-only hierarchical causal CNN tree'
    ),
    'accuracy': accuracy_score(test_true_labels, final_pred_labels),
    'balanced_accuracy': balanced_accuracy_score(test_true_labels, final_pred_labels),
    'macro_precision': precision_score(
        test_true_labels, final_pred_labels, labels=final_labels,
        average='macro', zero_division=0
    ),
    'macro_recall': recall_score(
        test_true_labels, final_pred_labels, labels=final_labels,
        average='macro', zero_division=0
    ),
    'macro_f1': f1_score(
        test_true_labels, final_pred_labels, labels=final_labels,
        average='macro', zero_division=0
    ),
    'weighted_f1': f1_score(
        test_true_labels, final_pred_labels, labels=final_labels,
        average='weighted', zero_division=0
    ),
    'n_test': len(test_true_labels),
}])

causal_tree_confusion_df = pd.DataFrame(
    confusion_matrix(test_true_labels, final_pred_labels, labels=final_labels),
    index=[f'true_{final_label_to_class[label]}' for label in final_labels],
    columns=[f'pred_{final_label_to_class[label]}' for label in final_labels],
)
causal_tree_confusion_normalized_df = causal_tree_confusion_df.div(
    causal_tree_confusion_df.sum(axis=1).replace(0, np.nan), axis=0
)

# Per-window predictions remain in memory for later inspection and visualization.
causal_tree_prediction_df = meta_causal_tree_df.loc[test_idx_all].copy().reset_index(drop=False)
causal_tree_prediction_df = causal_tree_prediction_df.rename(columns={'index': 'sample_index'})
causal_tree_prediction_df['true_label'] = test_true_labels
causal_tree_prediction_df['pred_label'] = final_pred_labels
causal_tree_prediction_df['true_class'] = causal_tree_prediction_df['true_label'].map(
    final_label_to_class
)
causal_tree_prediction_df['pred_class'] = causal_tree_prediction_df['pred_label'].map(
    final_label_to_class
)
causal_tree_prediction_df['stage1_event_probability'] = stage1_event_probability
causal_tree_prediction_df['node1_weather_probability'] = np.nan
causal_tree_prediction_df.loc[
    stage2_positions, 'node1_weather_probability'
] = node1_prob
causal_tree_prediction_df['node2_geopolitical_probability'] = np.nan
causal_tree_prediction_df.loc[
    node2_parent_positions, 'node2_geopolitical_probability'
] = node2_prob

if CAUSAL_TREE_USE_STAGE1:
    print('\n=== Stage 1 six-commodity unweighted mean (Table-5 comparable) ===')
    display(causal_stage1_macro_df)

    print('\n=== Stage 1 pooled metrics (used for routing diagnostics) ===')
    display(causal_stage1_pooled_df)

    print('\n=== Stage 1 per-commodity metrics ===')
    display(causal_stage1_metrics_df.sort_values('f1', ascending=False).reset_index(drop=True))

    print('\n=== Stage 1 confusion matrix ===')
    display(causal_stage1_confusion_df)

print('\n=== Sequential route sizes ===')
display(route_summary_df)

print('\n=== Stage-2 route quality and conditional node metrics ===')
display(causal_tree_node_metrics_df)

print(f'\n=== Final {len(final_labels)}-class metrics ===')
display(causal_tree_final_metrics_df)

print('\n=== Final per-class metrics ===')
display(causal_tree_per_class_df)

print('\n=== Final confusion matrix ===')
display(causal_tree_confusion_df)

print('\n=== Final row-normalized confusion matrix ===')
display(causal_tree_confusion_normalized_df.round(3))


In [ ]:
# Visualize representative predictions from the three-family hierarchical model.
# This cell reuses causal_tree_prediction_df and does not retrain any model.
import causal_tree_case_visualizer
reload(causal_tree_case_visualizer)
from causal_tree_case_visualizer import DEFAULT_CASES, render_causal_tree_examples

required_names = [
    'causal_tree_prediction_df', 'price_df',
    'causal_tree_events_by_commodity', 'causal_tree_events_df',
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        'Run the full causal hierarchical CNN pipeline first. '
        f'Missing: {missing_names}'
    )

# Each number selects the nth distinct commodity-event candidate.
CAUSAL_CASE_SAMPLE_NUMBER_BY_CASE = {
    'correct_weather': 1,
    'correct_geopolitical': 1,
    'correct_supply_macro': 2,
    'supply_macro_as_geopolitical': 1,
    'geopolitical_as_supply_macro': 1,
}

# Match the binary example figures while allowing case-specific legend placement.
CAUSAL_CASE_ANNOTATION_OPTIONS = {
    case_name: {'wrap_width': 34, 'legend_loc': 'best'}
    for case_name in DEFAULT_CASES
}

available_case_pairs = set(zip(
    causal_tree_prediction_df['true_label'].astype(int),
    causal_tree_prediction_df['pred_label'].astype(int),
))
available_causal_cases = {
    case_name: case_spec
    for case_name, case_spec in DEFAULT_CASES.items()
    if (case_spec['true_label'], case_spec['pred_label']) in available_case_pairs
}
unavailable_causal_cases = sorted(
    set(DEFAULT_CASES).difference(available_causal_cases)
)
if unavailable_causal_cases:
    print(f'Skipping unavailable prediction cases {unavailable_causal_cases}')

causal_tree_case_summary_df = render_causal_tree_examples(
    prediction_df=causal_tree_prediction_df,
    price_df=price_df,
    events_by_commodity=causal_tree_events_by_commodity,
    cases=available_causal_cases,
    sample_numbers=CAUSAL_CASE_SAMPLE_NUMBER_BY_CASE,
    annotation_options=CAUSAL_CASE_ANNOTATION_OPTIONS,
    plot_length_points=120,
    figsize=(8.8, 4.9),
    dpi=220,
)
display(causal_tree_case_summary_df)

# Summarize the combined supply-financial family by event structure and event name.
supply_financial_event_lookup_df = causal_tree_events_df.loc[
    causal_tree_events_df['category'].eq('supply_policy_macro_financial'),
    ['commodity', 'event_id', 'subcategory', 'event_structure_category'],
].drop_duplicates().rename(columns={'subcategory': 'event_name'})
supply_financial_event_accuracy_df = (
    causal_tree_prediction_df.loc[
        causal_tree_prediction_df['true_class'].eq('supply_policy_macro_financial')
    ]
    .merge(supply_financial_event_lookup_df, on=['commodity', 'event_id'], how='left')
    .assign(correct=lambda frame: frame['true_label'].eq(frame['pred_label']))
    .groupby(['event_structure_category', 'event_name'], dropna=False)
    .agg(
        test_windows=('sample_index', 'size'),
        correct_windows=('correct', 'sum'),
        correct_rate=('correct', 'mean'),
    )
    .sort_values(['correct_rate', 'test_windows'], ascending=[False, False])
    .reset_index()
)
print('Supply-policy and macro-financial event accuracy in the test split')
display(supply_financial_event_accuracy_df)


## Independent 2026 case study


In [ ]:
# Post-cutoff case study using the already trained hierarchical models.
# No fitting, threshold selection, or model updating occurs in this cell. Prediction
# dates begin at the cutoff, with the preceding L-1 observations retained as lookback.

required_names = [
    'DATA_DIR', 'predict_windows', 'causal_stage1_models', 'causal_tree_nodes',
    'causal_tree_commodities', 'causal_tree_scale_df', 'CAUSAL_TREE_SCALE_MODE',
    'PAPER_CUTOFF_DATE', 'L',
    'CAUSAL_TREE_BATCH_SIZE_EVAL', 'CAUSAL_TREE_DEVICE',
    'WEATHER_LABEL', 'GEOPOLITICAL_LABEL', 'SUPPLY_MACRO_LABEL',
    'CAUSAL_TREE_LABEL_TO_CATEGORY', 'PAPER_EVENT_FILE',
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        'Run the full event-detection and hierarchical CNN cell first. '
        f'Missing objects: {missing_names}'
    )
if not causal_stage1_models:
    raise RuntimeError('The case study requires the trained commodity-specific Stage-1 models.')

CASE_STUDY_CUTOFF_DATE = pd.Timestamp(PAPER_CUTOFF_DATE)
CASE_STUDY_WINDOW_LENGTH = int(L)
CASE_STUDY_PRICE_FILE = Path(DATA_DIR) / 'price.csv'
CASE_STUDY_EVENT_DATE = pd.Timestamp('2026-02-28')
CASE_STUDY_EVENT_LABEL = 'US–Israel strikes on Iran\nOutbreak of the 2026 Iran war'

case_price_df = pd.read_csv(CASE_STUDY_PRICE_FILE)
case_price_df['observation_date'] = pd.to_datetime(case_price_df['observation_date'])
case_price_df = case_price_df.set_index('observation_date').sort_index()
case_study_commodities = list(causal_tree_commodities)
missing_commodities = [c for c in case_study_commodities if c not in case_price_df.columns]
if missing_commodities:
    raise ValueError(f'Case-study prices are missing commodities: {missing_commodities}')

# Construct every complete window ending on or after the cutoff. This makes the first
# prediction window include 2026-02-20 and shifts the case-study sequence earlier.
case_X_parts = []
case_meta_rows = []
case_observation_rows = []
for commodity in case_study_commodities:
    full_series = case_price_df[commodity].dropna().sort_index()
    heldout_series = full_series.loc[full_series.index >= CASE_STUDY_CUTOFF_DATE]
    case_observation_rows.append({
        'commodity': commodity,
        'heldout_observations': len(heldout_series),
        'heldout_start': heldout_series.index.min() if len(heldout_series) else pd.NaT,
        'heldout_end': heldout_series.index.max() if len(heldout_series) else pd.NaT,
    })
    if heldout_series.empty or len(full_series) < CASE_STUDY_WINDOW_LENGTH:
        print(
            f'Skipping {commodity}. It has {len(heldout_series)} held-out observations, '
            f'fewer than the required {CASE_STUDY_WINDOW_LENGTH}.'
        )
        continue
    values = full_series.to_numpy(dtype=np.float32)
    dates = full_series.index
    first_cutoff_position = int(dates.searchsorted(CASE_STUDY_CUTOFF_DATE, side='left'))
    first_end_position = max(CASE_STUDY_WINDOW_LENGTH - 1, first_cutoff_position)
    for end_position in range(first_end_position, len(full_series)):
        start_position = end_position - CASE_STUDY_WINDOW_LENGTH + 1
        window = values[start_position:end_position + 1]
        if not np.isfinite(window).all():
            continue
        case_X_parts.append(window)
        case_meta_rows.append({
            'commodity': commodity,
            'window_start': dates[start_position],
            'window_end': dates[end_position],
        })

if not case_X_parts:
    raise ValueError('No complete case-study windows ending on or after the cutoff could be constructed.')

case_study_X_raw = np.stack(case_X_parts).astype(np.float32)
case_study_meta_df = pd.DataFrame(case_meta_rows)
case_study_meta_df.insert(0, 'case_sample_index', np.arange(len(case_study_meta_df)))
case_study_observation_df = pd.DataFrame(case_observation_rows)

# Stage 1 uses raw windows, matching its training specification. Stage 2 reuses only
# the commodity scales estimated from its training observations.
scale_lookup = causal_tree_scale_df.set_index('commodity')['positive_scale'].astype(float)
case_scales = case_study_meta_df['commodity'].map(scale_lookup)
if case_scales.isna().any():
    missing_scales = case_study_meta_df.loc[case_scales.isna(), 'commodity'].unique().tolist()
    raise ValueError(f'Missing fitted Stage-2 scales for commodities: {missing_scales}')
case_study_X_stage2 = (
    case_study_X_raw / case_scales.to_numpy(dtype=np.float32)[:, None]
).astype(np.float32)

# Stage 1 inference uses the corresponding trained model for each commodity.
case_stage1_pred = np.zeros(len(case_study_meta_df), dtype=int)
case_stage1_prob = np.full(len(case_study_meta_df), np.nan, dtype=float)
for commodity in case_study_commodities:
    if commodity not in causal_stage1_models:
        raise KeyError(f'No trained Stage-1 model is available for {commodity}.')
    commodity_positions = np.flatnonzero(
        case_study_meta_df['commodity'].astype(str).eq(str(commodity)).to_numpy()
    )
    commodity_pred, commodity_prob = predict_windows(
        causal_stage1_models[commodity], case_study_X_raw[commodity_positions],
        CAUSAL_TREE_DEVICE, batch_size=CAUSAL_TREE_BATCH_SIZE_EVAL,
    )
    case_stage1_pred[commodity_positions] = commodity_pred
    case_stage1_prob[commodity_positions] = commodity_prob

# Sequential Stage 2 routing. Stage-1 negatives stop at no_event.
case_final_labels = np.zeros(len(case_study_meta_df), dtype=int)
case_node1_pred = np.full(len(case_study_meta_df), np.nan)
case_node1_prob = np.full(len(case_study_meta_df), np.nan)
case_node2_pred = np.full(len(case_study_meta_df), np.nan)
case_node2_prob = np.full(len(case_study_meta_df), np.nan)

stage2_positions = np.flatnonzero(case_stage1_pred == 1)
node1_pred, node1_prob = predict_windows(
    causal_tree_nodes[1]['model'], case_study_X_stage2[stage2_positions],
    CAUSAL_TREE_DEVICE, batch_size=CAUSAL_TREE_BATCH_SIZE_EVAL,
)
case_node1_pred[stage2_positions] = node1_pred
case_node1_prob[stage2_positions] = node1_prob
node1_weather_mask = node1_pred == 1
case_final_labels[stage2_positions[node1_weather_mask]] = WEATHER_LABEL

node2_positions = stage2_positions[~node1_weather_mask]
node2_pred, node2_prob = predict_windows(
    causal_tree_nodes[2]['model'], case_study_X_stage2[node2_positions],
    CAUSAL_TREE_DEVICE, batch_size=CAUSAL_TREE_BATCH_SIZE_EVAL,
)
case_node2_pred[node2_positions] = node2_pred
case_node2_prob[node2_positions] = node2_prob
node2_geopolitical_mask = node2_pred == 1
case_final_labels[node2_positions[node2_geopolitical_mask]] = GEOPOLITICAL_LABEL
case_final_labels[node2_positions[~node2_geopolitical_mask]] = SUPPLY_MACRO_LABEL

case_label_to_class = {0: 'no_event', **CAUSAL_TREE_LABEL_TO_CATEGORY}
case_study_prediction_df = case_study_meta_df.copy()
case_study_prediction_df['stage1_event_probability'] = case_stage1_prob
case_study_prediction_df['stage1_pred_event'] = case_stage1_pred
case_study_prediction_df['node1_weather_probability'] = case_node1_prob
case_study_prediction_df['node1_prediction'] = case_node1_pred
case_study_prediction_df['node2_geopolitical_probability'] = case_node2_prob
case_study_prediction_df['node2_prediction'] = case_node2_pred
case_study_prediction_df['predicted_label'] = case_final_labels
case_study_prediction_df['predicted_class'] = pd.Series(case_final_labels).map(
    case_label_to_class
).to_numpy()

case_class_order = [
    'no_event',
    'weather_natural_hazard',
    'geopolitical_security',
    'supply_policy_macro_financial',
]
case_class_counts_df = (
    pd.crosstab(
        case_study_prediction_df['commodity'],
        case_study_prediction_df['predicted_class'],
    )
    .reindex(index=case_study_commodities, columns=case_class_order, fill_value=0)
)
case_study_summary_df = (
    case_study_prediction_df.groupby('commodity', sort=False)
    .agg(
        test_windows=('case_sample_index', 'size'),
        first_window_start=('window_start', 'min'),
        first_prediction_date=('window_end', 'min'),
        last_prediction_date=('window_end', 'max'),
        predicted_event_windows=('stage1_pred_event', 'sum'),
        predicted_event_rate=('stage1_pred_event', 'mean'),
        mean_event_probability=('stage1_event_probability', 'mean'),
    )
    .join(case_class_counts_df)
    .reset_index()
)

# Summarise consecutive predicted event runs without imposing unavailable true labels.
case_run_frames = []
for commodity, commodity_df in case_study_prediction_df.groupby('commodity', sort=False):
    commodity_df = commodity_df.sort_values('window_end').copy()
    commodity_df['run_id'] = (
        commodity_df['predicted_class'] != commodity_df['predicted_class'].shift()
    ).cumsum()
    run_df = (
        commodity_df.groupby(['run_id', 'predicted_class'], as_index=False)
        .agg(
            run_start=('window_end', 'min'),
            run_end=('window_end', 'max'),
            windows=('case_sample_index', 'size'),
            mean_event_probability=('stage1_event_probability', 'mean'),
        )
    )
    run_df.insert(0, 'commodity', commodity)
    case_run_frames.append(run_df)
case_study_runs_df = pd.concat(case_run_frames, ignore_index=True)
case_study_event_runs_df = case_study_runs_df.loc[
    case_study_runs_df['predicted_class'].ne('no_event')
].reset_index(drop=True)

# Audit the available curated table. It currently supplies no post-cutoff ground truth,
# so this cell reports model predictions rather than accuracy or a confusion matrix.
case_events_df = pd.read_excel(PAPER_EVENT_FILE)
for date_column in ['start_date', 'end_date', 'event_date']:
    if date_column in case_events_df.columns:
        case_events_df[date_column] = pd.to_datetime(
            case_events_df[date_column], errors='coerce'
        )
case_study_curated_events_df = case_events_df.loc[
    case_events_df['end_date'].ge(CASE_STUDY_CUTOFF_DATE),
    [
        column for column in [
            'event_id', 'commodity', 'start_date', 'end_date', 'event_date', 'category'
        ] if column in case_events_df.columns
    ],
].copy()

print('Post-cutoff case study with lookback context')
print(
    f'First prediction date {CASE_STUDY_CUTOFF_DATE.date()}, window length {CASE_STUDY_WINDOW_LENGTH}, '
    f'{len(case_study_prediction_df)} post-cutoff prediction windows, no retraining.'
)
print(f'Stage-2 scaling mode reused from training: {CAUSAL_TREE_SCALE_MODE}')
display(case_study_observation_df)
display(case_study_summary_df)
if case_study_event_runs_df.empty:
    print('No event runs were predicted in the held-out period.')
else:
    print('Consecutive predicted event runs')
    display(case_study_event_runs_df)
if case_study_curated_events_df.empty:
    print(
        'The current curated event table has no post-cutoff ground-truth events. '
        'These case-study outputs are predictions, not accuracy estimates.'
    )
else:
    print('Curated events intersecting the held-out period')
    display(case_study_curated_events_df)

# Plot from the start of the first lookback window. Each marker is the final class
# assigned to the L-observation window ending on that date. Nothing is written to disk.
case_class_colors = {
    'no_event': '#9E9E9E',
    'weather_natural_hazard': '#1F77B4',
    'geopolitical_security': '#D62728',
    'supply_policy_macro_financial': '#FF7F0E',
}
case_class_markers = {
    'no_event': 'o',
    'weather_natural_hazard': '^',
    'geopolitical_security': 's',
    'supply_policy_macro_financial': 'o',
}
case_class_hollow = {'supply_policy_macro_financial'}
case_study_plot_order = [
    'DCOILBRENTEU',
    'DHHNGSP',
    'DGASNYH',
    'DRGASLA',
    'DDFUELUSGULF',
    'DJFUELUSGULF',
]
if set(case_study_plot_order) != set(case_study_commodities):
    raise ValueError('The case-study plot order must contain each selected commodity once.')
fig, axes = plt.subplots(
    len(case_study_plot_order), 1,
    figsize=(12, 2.8 * len(case_study_plot_order)),
    sharex=True,
)
axes = np.atleast_1d(axes)
for axis_index, (axis, commodity) in enumerate(zip(axes, case_study_plot_order)):
    commodity_predictions = case_study_prediction_df.loc[
        case_study_prediction_df['commodity'].eq(commodity)
    ]
    plot_start = commodity_predictions['window_start'].min()
    heldout_prices = (
        case_price_df.loc[case_price_df.index >= plot_start, commodity]
        .dropna()
        .sort_index()
    )
    axis.plot(
        heldout_prices.index, heldout_prices.to_numpy(),
        color='black', linewidth=1.2, alpha=0.75,
    )
    for class_name in case_class_order:
        class_predictions = commodity_predictions.loc[
            commodity_predictions['predicted_class'].eq(class_name)
        ]
        if class_predictions.empty:
            continue
        marker_prices = heldout_prices.reindex(class_predictions['window_end']).to_numpy()
        marker_color = case_class_colors[class_name]
        axis.scatter(
            class_predictions['window_end'], marker_prices,
            s=30, marker=case_class_markers[class_name],
            facecolors='none' if class_name in case_class_hollow else marker_color,
            edgecolors=marker_color, linewidths=1.0,
            label=class_name, zorder=3,
        )
    axis.axvline(
        CASE_STUDY_EVENT_DATE, color='#6A3D9A', linestyle='--',
        linewidth=1.4, alpha=0.90, zorder=2,
    )
    if axis_index == 0:
        axis.annotate(
            CASE_STUDY_EVENT_LABEL,
            xy=(CASE_STUDY_EVENT_DATE, 1.0),
            xycoords=('data', 'axes fraction'),
            xytext=(-6, -6), textcoords='offset points',
            ha='right', va='top', fontsize=9, color='#6A3D9A',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=2),
        )
    axis.set_title(commodity, loc='left', pad=8.0)
    axis.set_ylabel('Price')
    axis.grid(alpha=0.20)
axes[-1].set_xlabel('Window end date')
legend_handles = [
    plt.Line2D(
        [0], [0], color='black', linewidth=1.2, label='Price',
    ),
]
legend_handles.extend([
    plt.Line2D(
        [0], [0], marker=case_class_markers[class_name],
        linestyle='', markersize=7,
        markerfacecolor=(
            'none' if class_name in case_class_hollow
            else case_class_colors[class_name]
        ),
        markeredgecolor=case_class_colors[class_name],
        color=case_class_colors[class_name], label=class_name,
    )
    for class_name in case_class_order
])
legend_handles.append(
    plt.Line2D(
        [0], [0], color='#6A3D9A', linestyle='--', linewidth=1.4,
        label='28 Feb 2026 — outbreak of the 2026 Iran war',
    )
)
fig.legend(
    handles=legend_handles, loc='upper center', ncol=2,
    bbox_to_anchor=(0.5, 0.995), frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.94), h_pad=1.5)
plt.show()
